In [ ]:
# author: astarosina
# updated: 20251229

# Intro

#### Prerequisites
* access to Trino tech_dataplatform_core.system_queries_stat
* file for dbt-dbt_datalake_manifest.json
* token for data hub API

# Step 0. Trino connection

In [ ]:
from trino.dbapi import connect
from trino.auth import BasicAuthentication
import urllib3
from urllib3.exceptions import InsecureRequestWarning
import credscram
import pandas as pd
import re

urllib3.disable_warnings(InsecureRequestWarning)

conn = connect(
    host="trino.exness.io",
    port=443,
    auth=BasicAuthentication(credscram.creds.ldap_name, credscram.creds.ldap_pass),
    http_scheme="https",
    verify=False,
)


# Stage 1. Get tables metadata from Trino

* Get the list of delta and iceberg tables and views from information schema and then enrich tables with info about DDL (`delta.tech_data_platform_core.trino_tables_metadata`) and last updated at (`delta.tech_data_platform_core.delta_log_stats`).

* Instert the output into `delta.tech_data_platform_work.astarostina_master_table_metadata`

In [ ]:
# updated 20251229
CATALOGS_TO_CHECK = ["delta", "iceberg"]

EXCLUDE_SCHEMAS = {
    "trading_work","tech_ml_platform_work","partnership_work","marketing_work","finance_work",
    "protection_work","trading_pricing_work","information_schema","system","common_uploads",
    "commercial_work","tech_data_platform_work","payments_work", "relationship_work", "tech_data_platform_dbt_tmp_mart"
}
TARGET_TABLE = "delta.tech_data_platform_work.astarostina_master_table_metadata"

CREATE_TABLE_SQL = f"""
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
    catalog             varchar,
    schema              varchar,
    table_name          varchar,
    table_type          varchar,   -- BASE TABLE / VIEW

    columns             varchar,   -- comma-separated column names
    partitioned_by      varchar,   -- NULL if not meaningful

    last_data_update    timestamp
)
"""

cur.execute(CREATE_TABLE_SQL)
print("[ok] master_table_metadata exists (or has been created)")

#--- Insert Overwrite

excluded_schemas_sql = ", ".join(f"'{s}'" for s in sorted(EXCLUDE_SCHEMAS))

MASTER_TABLE_METADATA_SQL = f"""
CREATE OR REPLACE TABLE {TARGET_TABLE} AS 

WITH inventory AS (

    -- ===== DELTA =====
    SELECT
        'delta'                 AS catalog,
        lower(table_schema)     AS schema,
        lower(table_name)       AS table_name,
        table_type
    FROM delta.information_schema.tables
    WHERE table_type IN ('BASE TABLE', 'VIEW')
      AND lower(table_schema) NOT IN ({excluded_schemas_sql})

    UNION ALL

    -- ===== ICEBERG =====
    SELECT
        'iceberg'               AS catalog,
        lower(table_schema)     AS schema,
        lower(table_name)       AS table_name,
        table_type
    FROM iceberg.information_schema.tables
    WHERE table_type IN ('BASE TABLE', 'VIEW')
      AND lower(table_schema) NOT IN ({excluded_schemas_sql})
),

ddl_enriched AS (
    SELECT
        t.catalog,
        t.schema,
        t."table" AS table_name,

        -- comma-separated column list
        array_join(
            transform(
                split(
                    regexp_extract(
                        t.create_table_ddl,
                        '(?is)create\\s+table.*?\\((.*?)\\)\\s*with',
                        1
                    ),
                    ','
                ),
                c -> trim(split(trim(c), ' ')[1])
            ),
            ', '
        ) AS columns,

        -- meaningful partitioning only
        CASE
            WHEN partition_cols = ARRAY['millennium'] THEN NULL
            ELSE array_join(partition_cols, ', ')
        END AS partitioned_by
    FROM (
        SELECT
            catalog,
            schema,
            "table",
            create_table_ddl,
            transform(
                split(
                    regexp_extract(
                        create_table_ddl,
                        '(?i)partitioned_by\\s*=\\s*array\\s*\\[(.*?)\\]',
                        1
                    ),
                    ','
                ),
                p -> lower(trim(replace(replace(p, '''', ''), '"', '')))
            ) AS partition_cols
        FROM delta.tech_data_platform_core.trino_tables_metadata
    ) t
),

freshness AS (
    SELECT
        catalog,
        schema,
        "table" AS table_name,
        max(last_version_dt) AS last_data_update
    FROM delta.tech_data_platform_core.delta_log_stats
    GROUP BY
        catalog,
        schema,
        "table"
)

SELECT
    i.catalog,
    i.schema,
    i.table_name,
    i.table_type,
    d.columns,
    d.partitioned_by,
    f.last_data_update
FROM inventory i
LEFT JOIN ddl_enriched d
    ON d.catalog    = i.catalog
   AND d.schema     = i.schema
   AND d.table_name = i.table_name
LEFT JOIN freshness f
    ON f.catalog    = i.catalog
   AND f.schema     = i.schema
   AND f.table_name = i.table_name
"""

cur.execute(MASTER_TABLE_METADATA_SQL)
print("[done] master_table_metadata rebuilt successfully")



# Stage 2. Get tables usage info from Trino

## Step 2.1. Select query and user stat from system_runtime_queries

In [ ]:
# =============================================================================
# Load + Normalize Trino runtime queries into a Delta table (idempotent by query_id + created_dt)
# - Arbitrary period load (FROM_DT, TO_DT)
# - Safe refresh: DELETE target rows for the period by (query_id, created_dt) then INSERT
# - Extract Metabase header fields + normalize query text
# =============================================================================

# -----------------------------
# Configuration
# -----------------------------
TARGET_TABLE = "delta.tech_data_platform_work.astarostina_trino_analytical_queries"

# Arbitrary period (inclusive start, exclusive end)
FROM_DT = "2025-01-01"
TO_DT   = "2026-01-01"  # exclusive

# -----------------------------
# 0) Create target table if not exists (with created_dt + query_id)
# -----------------------------
create_table_sql = f"""
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
    query_id VARCHAR,
    created_dt DATE,

    query_norm VARCHAR,
    u VARCHAR,

    metabase_account_id VARCHAR,
    metabase_user_id BIGINT,
    metabase_query_type VARCHAR,
    metabase_card_id BIGINT,
    metabase_dashboard_id BIGINT,

    request_type VARCHAR,

    etl_updated_at TIMESTAMP
)
"""
cursor = conn.cursor()
cursor.execute(create_table_sql)

# -----------------------------
# 1) Delete existing rows for the input period (idempotent refresh)
# -----------------------------
delete_sql = f"""
DELETE FROM {TARGET_TABLE}
WHERE (query_id, created_dt) IN (
    SELECT
        query_id,
        CAST(created_dt AS DATE) AS created_dt
    FROM delta.tech_data_platform_core.system_runtime_queries
    WHERE created_dt >= DATE '{FROM_DT}'
      AND created_dt <  DATE '{TO_DT}'
      AND state = 'FINISHED'
)
"""
cursor.execute(delete_sql)

# -----------------------------
# 2) Regex fragments (Trino SQL expressions)
# -----------------------------
METABASE_ACCOUNT_ID_REGEXP   = "regexp_extract(query_lc, '(?i)accountid\\s*[:=]\\s*([0-9a-f-]{36})', 1)"
METABASE_USER_ID_REGEXP      = "try_cast(regexp_extract(query_lc, '(?i)userid\\s*[:=]\\s*(\\d+)', 1) AS bigint)"
METABASE_QUERY_TYPE_REGEXP   = "regexp_extract(query_lc, '(?i)querytype\\s*[:=]\\s*([a-z0-9_]+)', 1)"
METABASE_DASHBOARD_ID_REGEXP = "try_cast(regexp_extract(query_lc, '(?i)dashboardid\\s*[:=]\\s*(\\d+)', 1) AS bigint)"
METABASE_CARD_ID_REGEXP      = "try_cast(regexp_extract(query_lc, '(?i)cardid\\s*[:=]\\s*(\\d+)', 1) AS bigint)"

# -----------------------------
# 3) Query normalization chain (Trino regexp_replace chain)
#
# Notes:
# - Input is already lower()'d into query_lc
# - We:
#   * unwrap EXECUTE IMMEDIATE '...'
#   * remove metabase header lines/comments
#   * normalize date/timestamp literals into placeholders
#   * normalize UID/UUID IN(...) and = ... patterns (only columns containing uid/uuid or uuid itself)
#   * collapse whitespace
#   * normalize Metabase substring aliases:  ... as "substring12345" -> {{substring_alias}}
#   * normalize LIMIT n -> LIMIT 10
# -----------------------------
CLEANING_CHAIN = r"""
lower(
  trim(
    regexp_replace( -- metabase temp_dir
     regexp_replace( -- metabase substring aliases
      regexp_replace( -- limit
        regexp_replace( -- collapse spaces
          regexp_replace( -- collapse new lines
            regexp_replace( -- remove everything before first SQL keyword
              regexp_replace( -- remove leading comment lines (safe after unwrapping)
              regexp_replace( -- normalize UID/UUID equality - simple
               /*  regexp_replace( -- normalize UID/UUID equality
                  regexp_replace( -- normalize cast(uid..) in (...)
                    regexp_replace( -- normalize uid/uuid in (...)
                     */ 
                     regexp_replace( -- uuids
                     regexp_replace( -- normalize from_iso8601_timestamp('...')
                        regexp_replace( -- normalize parse_datetime('...','...')
                          regexp_replace( -- normalize timestamp 'YYYY-MM-DD ...'
                          regexp_replace( -- normalize timestamp 'YYYY-MM-DD ...'
                          regexp_replace( -- normalize timestamp 'YYYY-MM-DD ...'
                            regexp_replace( -- normalize cast('...' as timestamp)
                              regexp_replace( -- normalize date(cast('...' as timestamp))
                                regexp_replace( -- normalize cast('YYYY-MM-DD' as date)
                                  regexp_replace( -- normalize date 'YYYY-MM-DD'
                                    regexp_replace( -- normalize date('YYYY-MM-DD') (and weird date ''2025-06-01'')
                                      regexp_replace( -- normalize bare 'YYYY-MM-DD' (quoted date literals)
                                        regexp_replace( -- strip metabase accountid comment
                                          (
                                            CASE
                                              WHEN regexp_like(query_lc, '(?is)^\s*execute\s+immediate\s*''')
                                                THEN regexp_extract(query_lc, '(?is)^\s*execute\s+immediate\s*''((?:[^'']|'''')*)''', 1)
                                              ELSE query_lc
                                            END
                                          ),
                                          '(?is)--\s*metabase::.*?(?=\b(select|with|insert|update|delete|merge|call|values)\b)', ' '
                                      ),
                                      '''\d{4}-\d{2}-\d{2}''', '{{date_value}}'
                                      ),
                                      'date\s*\(\s*''+\d{4}-\d{2}-\d{2}''+\s*\)', '{{date_value}}'
                                    ),
                                    'date\s*''\d{4}-\d{2}-\d{2}''', '{{date_value}}'
                                  ),
                                  'cast\s*\(\s*''\d{4}-\d{2}-\d{2}''\s+as\s+date\s*\)', '{{date_value}}'
                                ),
                                'date\s*\(\s*cast\s*\(\s*''[^'']+''\s+as\s+timestamp\s*\)\s*\)', '{{timestamp_value}}'
                              ),
                              'cast\s*\(\s*''[^'']+''\s+as\s+timestamp\s*\)', '{{timestamp_value}}'
                            ),
                            'timestamp\\s*''\\d{4}-\\d{2}-\\d{2}(?:\\s+\\d{2}:\\d{2}:\\d{2}(?:\\.\\d+)?)?''', '{{timestamp_value}}' 
                         ), '(?i)timestamp\s*''\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}(?:\.\d+)?\s*[+-]\d{2}:\d{2}''', '{{timestamp_value}}' 
  
                          ), 'timestamp\s*''\d{4}-\d{2}-\d{2}(?:\s+\d{2}:\d{2}:\d{2}(?:\.\d+)?)?''', '{{timestamp_value}}' 
                          
                         ), 'parse_datetime\s*\(\s*''[^'']+''\s*,\s*''[^'']+''\s*\)', '{{timestamp_value}}'
                        ),
                        'from_iso8601_timestamp\s*\(\s*''[^'']+''\s*\)', '{{timestamp_value}}'
                      )
                      , '(?i)(?:''[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}''\s*,\s*)+''[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}''',
  '{{uuids}}')
                      -- uid/uuid in (...)  (supports nested parentheses; bounded enough for Trino)
                    /* , '(?is)(\b(?:[\w]+\.)?(?:\w*uid\w*|\w*uuid\w*|uuid)\b)\s+in\s*\((?:[^()]*|\([^()]*\))*\)',
                      '$1 in {{uuids}}'
                    ),
                    -- cast(uid...) in (...)
                    '(?is)\bcast\s*\(\s*([\w\.]*?(?:uid|uuid)[\w\.]*)\s+as\s+\w+\s*\)\s+in\s*\((?:[^()]*|\([^()]*\))*\)',
                    '$1 in {{uuids}}'
                  ),
                  -- uid/uuid = ? / 'uuid' / cast(...)
                  '(?is)(\b(?:[\w]+\.)?(?:\w*uid\w*|\w*uuid\w*|uuid)\b)\s*=\s*(?:\?|cast\([^)]*\)|''+[0-9a-f-]{36}''+)',
                  '$1 = {{uuid}}')
                */
                , '(?i)(=)\s*''[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}''','{{uuid}}')
                
                , '^(\s*--[^\r\n]*\r?\n)+', ' '
              ),'(?is)^.*?(?=\b(select|with|insert|update|delete|merge|call|values)\b)', ''
            ),'\r?\n+', ' '
          ),'\s+', ' '
        ),'(?i)\blimit\s+\d+\b', 'limit 10'
        
        -- metabase specific 
      ), '\s+as\s+\"substring\d+\"', ' as {{substring_alias}}' 
      ), '(?i)\btmp_dir_[0-9a-f]+\b', 'tmp_dir' -- metabase specific 
   )
  )
)
"""

# -----------------------------
# 4) Insert load SQL (arbitrary period; idempotent by query_id+created_dt)
# -----------------------------
insert_sql = f"""
INSERT INTO {TARGET_TABLE}
WITH
params AS (
    SELECT
        DATE '{FROM_DT}' AS from_dt,
        DATE '{TO_DT}'   AS to_dt
),
events AS (
    SELECT
        query_id,
        CAST(created_dt AS DATE)         AS created_dt,
        created_dt                       AS created_ts,
        "user"                           AS u,
        lower(query)                     AS query_lc,
        query                            AS query_raw,
        source
    FROM delta.tech_data_platform_core.system_runtime_queries t
    WHERE created_dt >= (SELECT from_dt FROM params)
      AND created_dt <  (SELECT to_dt   FROM params)
      AND state = 'FINISHED'
      AND "user" NOT IN (
            'kirill.mironov','pavel.semenov','pavel.zakharov',
            'artur.burlaka','iurii.grabovskii','a.starostina',
            'stanislav.vovk','ed.avetisyan','ivan.mostovich',
            'artem.buzin','alexey.izotov','anton.shusharin','nikolai.grishchenkov'
      )
      AND t."user" NOT IN (
            'app_s3_tickl','app_s3_trino','app_s3_trinoaf',
            'app_s3_trino-dq','app_s3_catalog','app_s3_flink',
            'app_s3_af-dbt-gen-prod', 'app_s3_transcrb-api', 'app_s3_charon'
      )
),
classified AS (
    SELECT
        *,
        CASE
            WHEN u LIKE 'app_%' THEN 'app'
            WHEN regexp_like(trim(u), '^[a-z]+[.][a-z]+$') THEN 'user_request'
            WHEN u IN ('anton.medvedev2') THEN 'user_request'
            ELSE u
        END AS request_type,
        CASE
            -- noise / metadata
            WHEN query_lc LIKE 'insert into%' 
                OR query_lc LIKE 'update %'
                OR query_lc LIKE 'delete %'
                OR query_lc LIKE 'merge into%' THEN 'data_mutation'
                
            WHEN query_lc LIKE 'create or replace table%'
                OR query_lc LIKE 'create table%'
                OR query_lc LIKE 'alter table%'
                OR query_lc LIKE 'drop%'
                OR query_lc LIKE 'truncate%' THEN 'ddl'
    
            WHEN query_raw LIKE 'SELECT  CAST(NULL AS varchar) TABLE_CAT%' 
                OR query_lc LIKE '%select table_cat%'
                OR query_lc LIKE '%deferrability%' 
                OR source LIKE '%Metadata%'
                OR query_lc LIKE '%describe%'
                OR query_lc LIKE '%show columns%'
                OR query_lc LIKE '%show schemas from%' 
                OR query_lc LIKE '%show create table%'
                OR query_lc LIKE '%show tables from%'
                OR query_lc LIKE '%select partition%'
                THEN 'system_search'
                
            WHEN query_lc LIKE '%select procedure_cat%' THEN 'noise'
            WHEN query_lc LIKE '%select current_user%' THEN 'noise'
            WHEN query_lc LIKE '%select ''keep alive''%' THEN 'noise'
            WHEN query_lc LIKE '%select version()%' THEN 'noise'
            
            WHEN query_lc LIKE '%system.runtime.queries%' THEN 'noise'
            WHEN query_lc LIKE '%select 1%' THEN 'noise'
            WHEN query_lc LIKE '%start transaction%' THEN 'noise'
            WHEN query_lc LIKE '%set session query_max_stage_count%' THEN 'noise'
            WHEN query_lc IN ('commit', 'rollback') THEN 'noise'

            WHEN query_lc LIKE '%information_schema%' THEN 'noise'
            WHEN query_lc LIKE '%system.jdbc%' THEN 'noise'
            WHEN query_lc LIKE 'select current_timezone() as "time-zone"' THEN 'noise'
            WHEN query_lc LIKE '%from "common_uploads".%' THEN 'noise'

            WHEN query_raw LIKE 'PREPARE ibis_trino_metadata%' THEN 'noise'
            WHEN query_lc LIKE '%execute st_%' THEN 'noise' -- app_s3_idash
            WHEN query_lc LIKE '%set session iceberg.expire_snapshots_min_retention%' THEN 'noise' -- dbt 

            -- "real" queries heuristics
            WHEN query_lc LIKE '%delta.%' THEN 'real'
            WHEN query_lc LIKE '%"delta".%' THEN 'real'
            WHEN query_lc LIKE '%iceberg.%' THEN 'real'
            WHEN query_lc LIKE '%"iceberg".%' THEN 'real'
            WHEN query_lc LIKE '%with%' THEN 'real'
            WHEN query_lc LIKE '%commercial%' THEN 'real'
            WHEN query_lc LIKE '%finance%' THEN 'real'
            WHEN query_lc LIKE '%trading%' THEN 'real'
            WHEN query_lc LIKE '%people_core%' THEN 'real'
            WHEN query_lc LIKE '%payments_%' THEN 'real'
            WHEN query_lc LIKE '%pricing_core%' THEN 'real'
            WHEN query_lc LIKE '%protection_%' THEN 'real'
            WHEN query_lc LIKE '%partnership_%' THEN 'real'
            WHEN query_lc LIKE '%marketing_%' THEN 'real'
            WHEN query_lc LIKE '%relationship_%' THEN 'real'
            WHEN query_lc LIKE '%tech_data_platform_core%' THEN 'real'
            WHEN query_lc LIKE '%translation%' THEN 'real'
            WHEN query_lc LIKE '%from clickhouse%' THEN 'real'
            WHEN query_lc LIKE '%from vertica%' THEN 'real'

            WHEN query_lc NOT LIKE '%from %' THEN 'noise'
            ELSE 'unknown'
        END AS query_category
    FROM events
),
norm AS (
    SELECT
        query_id,
        created_dt,
        created_ts,
        u,
        request_type,
        -- cleaned text
        {CLEANING_CHAIN} AS query_clean_norm,

        -- metabase fields (only for metabase users)
        CASE WHEN u LIKE '%metabase%' THEN {METABASE_ACCOUNT_ID_REGEXP} END AS metabase_account_id,
        CASE WHEN u LIKE '%metabase%' THEN {METABASE_USER_ID_REGEXP} END    AS metabase_user_id,
        CASE WHEN u LIKE '%metabase%' THEN {METABASE_QUERY_TYPE_REGEXP} END AS metabase_query_type,
        CASE WHEN u LIKE '%metabase%' THEN {METABASE_CARD_ID_REGEXP} END    AS metabase_card_id,
        CASE WHEN u LIKE '%metabase%' THEN {METABASE_DASHBOARD_ID_REGEXP} END AS metabase_dashboard_id
    FROM classified
    WHERE query_category in ('real', 'unknown')
)
   
SELECT
    query_id,
    created_dt,
    query_clean_norm AS query_clean,
    u,
    metabase_account_id,
    metabase_user_id,
    metabase_query_type,
    metabase_card_id,
    metabase_dashboard_id,
    request_type,
    current_timestamp AS etl_updated_at
FROM norm
"""

cursor.execute(insert_sql)
print(f"[done] Loaded period {FROM_DT}..{TO_DT} into {TARGET_TABLE} (refreshed by query_id+created_dt).")


## Step 2.2 Derive tables and stat, samples, top joins, top columns

In [ ]:
#### new 20251229

In [ ]:
# ================================================================
# STREAMING, LOW-MEM TRINO VERSION
# Builds table usage stats without loading everything into RAM.
# ================================================================

import re
import hashlib
from collections import Counter, defaultdict

import pandas as pd

# ---------------- CONFIG ----------------

TRINO_INPUT_TABLE   = "delta.tech_data_platform_work.astarostina_trino_analytical_queries"
OUTPUT_CSV = "2_table_stats.csv"

CHUNKSIZE     = 250_000
MAX_QUERY_LEN = 2500

TOP_K_JOINS   = 15
TOP_K_COLS    = 20
TOP_K_QUERIES = 5



# ---- EXCLUSIONS ----
EXCLUDE_JOIN_SCHEMAS = {
    "trading_work","tech_ml_platform_work","trading_pricing_work","tech_data_platform_work",
    "partnership_work","marketing_work","finance_work","protection_work","commercial_work",
    "payments_work","common_uploads","jdbc", "tech_data_platform_dbt_tmp_mart"
}

# ---------------- REGEX & HELPERS ----------------

def _strip_quotes(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.strip().replace('"','').replace('`','')
    return re.sub(r'^\[|\]$', '', s)

def _norm_ident(s: str) -> str:
    s = _strip_quotes(s)
    s = re.sub(r'\s+', ' ', s).strip().lower()
    return re.sub(r'[^a-z0-9._]', '', s)

CTE_NAME_RE     = re.compile(r'(?is)\b([`"\[]?[A-Za-z_][A-Za-z0-9_]*[`"\]]?)\s+as\s*\(')
TABLE_CLAUSE_RE = re.compile(r'(?is)\b(from|join|into|update|merge\s+into|table)\s+([`"\[\]A-Za-z0-9_.]+)')
JOIN_TARGET_RE  = re.compile(r'(?is)\bjoin\s+([`"\[\]A-Za-z0-9_.]+)')
COMMENT_RE     = re.compile(r"(--[^\n]*\n)|(/\*.*?\*/)", re.S | re.I)

SKIP_QUERY_RE = re.compile(
    r'''(?is)^\s*(?:--[^\n]*\n\s*|/\*.*?\*/\s*)*
        (?:create\s+(?:or\s+replace\s+)?(?:table|view)\b
           |insert\s+(?:overwrite\s+)?into\b)''',
    re.VERBOSE
)

EXCLUDE_NAMES    = {'unnest','lateral','values','if','case','true','false','null'}
EXCLUDE_PREFIXES = ('information_schema.', 'system.', 'sys.', 'sqlite_', '__temp__', '_tmp_', 'tmp_')

# ---------------- SQL PARSING ----------------

def _is_ddl_dml_to_skip(sql: str) -> bool:
    return bool(sql and SKIP_QUERY_RE.search(sql))

def extract_cte_names(sql: str) -> set:
    if not isinstance(sql, str) or not sql.strip():
        return set()
    if not re.search(r'(?is)^\s*with\b', sql):
        return set()
    return {_norm_ident(m.group(1)) for m in CTE_NAME_RE.finditer(sql)}

def looks_like_table(name: str) -> bool:
    if '.' not in name:
        return False
    if any(name.startswith(p) for p in EXCLUDE_PREFIXES):
        return False
    if name in EXCLUDE_NAMES:
        return False
    return True

def extract_tables(sql: str) -> set:
    ctes = extract_cte_names(sql)
    found = set()
    for m in TABLE_CLAUSE_RE.finditer(sql or ""):
        ident = _norm_ident(m.group(2))
        if ident and ident not in ctes and looks_like_table(ident):
            found.add(ident)
    return found

def extract_join_targets(sql: str) -> set:
    out = set()
    for m in JOIN_TARGET_RE.finditer(sql or ""):
        ident = _norm_ident(m.group(1))
        if ident and looks_like_table(ident):
            out.add(ident)
    return out

def _to_base_table(ident: str) -> str | None:
    parts = ident.split('.')
    if len(parts) == 3:
        return f"{parts[1]}.{parts[2]}"
    if len(parts) == 2:
        return ident
    return None

def _schema_of_base(bt: str) -> str | None:
    return bt.split('.', 1)[0] if '.' in bt else None

def find_columns_for_table(sql: str, base_table: str) -> list:
    try:
        schema, tbl = base_table.split('.', 1)
    except ValueError:
        return []
    return re.findall(
        rf'(?i)\b(?:{schema}\s*\.\s*)?{tbl}\s*\.\s*([a-z_][a-z0-9_]*)\b',
        sql or ""
    )

def _strip_sql_comments(sql: str) -> str:
    return COMMENT_RE.sub(" ", sql or "")

def _is_trivial_for_base(sql: str, base_table: str) -> bool:
    s = _strip_sql_comments(sql).lower()
    if re.search(r'\b(where|join|group\s+by|having|union|window)\b', s):
        return False
    if re.search(r'\b(count|sum|avg|min|max)\s*\(|\bover\b', s):
        return False
    tables = {_to_base_table(t) for t in extract_tables(sql) if _to_base_table(t)}
    return tables == {base_table}

def _normalize_and_truncate_query(q: str) -> str:
    q = re.sub(r'\s+', ' ', q or "").strip()
    return q[:MAX_QUERY_LEN]

def hash64(s: str) -> bytes:
    return hashlib.blake2b(s.encode("utf-8"), digest_size=8).digest()

# ---------------- ACCUMULATORS ----------------

join_counter     = defaultdict(Counter)
join_kw_counter  = defaultdict(Counter)
column_counter   = defaultdict(Counter)
sample_counter   = defaultdict(Counter)

total_weight_per_bt = defaultdict(int)
users_set        = defaultdict(set)
query_hash_set   = defaultdict(set)
catalogs_set     = defaultdict(set)

# ---------------- TRINO STREAM ----------------


SQL = f"""
SELECT
    query_norm as query,
    u as "user",
    count(*) AS num_queries
FROM {TRINO_INPUT_TABLE}
group by 1,2
"""

df_iter = pd.read_sql_query(SQL, conn, chunksize=CHUNKSIZE)

# ---------------- MAIN LOOP ----------------

total_rows = 0

for chunk in df_iter:
    chunk = chunk.dropna(subset=["query"]).drop_duplicates(subset=["query","user"])

    for q, u, w in zip(chunk["query"], chunk["user"], chunk["num_queries"]):
        total_rows += 1
        w = int(w) if w and w > 0 else 1

        if _is_ddl_dml_to_skip(q):
            continue

        tables = extract_tables(q)
        joins  = extract_join_targets(q)
        if not tables:
            continue

        base_set = {_to_base_table(t) for t in tables if _to_base_table(t)}
        base_set = {b for b in base_set if _schema_of_base(b) not in EXCLUDE_JOIN_SCHEMAS}
        if not base_set:
            continue

        for t in tables:
            p = _norm_ident(t).split(".")
            if len(p) == 3 and p[0] in {"delta","iceberg"}:
                catalogs_set[f"{p[1]}.{p[2]}"].add(p[0])

        q_short = _normalize_and_truncate_query(q)

        for bt in base_set:
            total_weight_per_bt[bt] += w
            if u:
                users_set[bt].add(u)

            for o in base_set - {bt}:
                join_counter[bt][o] += w

            for o in {_to_base_table(j) for j in joins if _to_base_table(j)} - {bt}:
                join_kw_counter[bt][o] += w

            if not _is_trivial_for_base(q, bt):
                for c in find_columns_for_table(q, bt):
                    column_counter[bt][c] += w
                sample_counter[bt][q_short] += w
                query_hash_set[bt].add(hash64(q_short))

    if total_rows % 500_000 == 0:
        print(f"[progress] processed ~{total_rows:,} rows")

# ---------------- BUILD OUTPUT ----------------

def top_list(cnt: Counter, k: int):
    return [x for x, _ in cnt.most_common(k)]

rows = []
for bt in sorted(set(users_set)):
    rows.append({
        "base_table": bt,
        "queries_cnt": len(query_hash_set[bt]),
        "queries_cnt_w": total_weight_per_bt[bt],
        "users_cnt": len(users_set[bt]),
        "users_list": sorted(users_set[bt]),
        "catalogs": sorted(catalogs_set.get(bt, [])),
        "top_joins": top_list(join_kw_counter[bt] or join_counter[bt], TOP_K_JOINS),
        "top_columns": top_list(column_counter[bt], TOP_K_COLS),
        "sample_queries": top_list(sample_counter[bt], TOP_K_QUERIES),
    })

out = pd.DataFrame(rows)
out.to_csv(OUTPUT_CSV, index=False)

print(f"[done] Saved {OUTPUT_CSV} | rows={len(out)} | processed={total_rows:,}")


## Step 2.3 Enrich with info about departments and divisions used by

In [ ]:
# === Enrich table stats with user org attributes (jobs/departments/directions/managers) ===
import re, json
import pandas as pd

INPUT_TABLES_CSV    = "2_table_stats.csv"                     # has base_table, users_list
OUTPUT_ENRICHED_CSV = "2_table_stats_final.csv"

def _to_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        # try JSON (double quotes). If it's single-quoted, fall back to split.
        try:
            v = json.loads(s)
            if isinstance(v, list):
                return v
        except Exception:
            pass
        # fallback: split by comma, trim brackets
        s = s.strip("[]")
        parts = [p.strip() for p in s.split(",") if p.strip()]
        return parts
    return []

UID_RE = re.compile(r"^\s*['\"]?(?P<uid>[^'\",@\s]+(?:\.[^'\",@\s]+)?)['\"]?\s*$")
def normalize_uid(val: str) -> str:
    """strip quotes/whitespace, take left of '@', force lower; keep name.surname only"""
    if val is None:
        return ""
    s = str(val).strip()
    # cut domain if email
    if "@" in s:
        s = s.split("@", 1)[0]
    m = UID_RE.match(s)
    if not m:
        return s.strip(" '\"").lower()
    return m.group("uid").lower()

# 1) Load base tables and normalize users_list -> list of clean uids
tbl = pd.read_csv(INPUT_TABLES_CSV, low_memory=False).copy()
if "users_list" not in tbl.columns:
    raise ValueError("Input must contain 'users_list' column.")

tbl["users_list"] = tbl["users_list"].apply(_to_list).apply(lambda lst: [normalize_uid(u) for u in lst if normalize_uid(u)])

# 2) Pull user directory
cursor = conn.cursor()
sql_users = """
SELECT uid, job, direction, department, group_job, uid_manager
FROM (
    SELECT 
        SPLIT_PART(ep.label, '@', 1) AS uid
      , max(CASE WHEN object_type_attribute_id = 458 THEN c.text_value END) AS job
      , max(CASE WHEN object_type_attribute_id = 456 THEN c.text_value END) AS direction
      , max(CASE WHEN object_type_attribute_id = 457 THEN c.text_value END) AS department
      , max(CASE WHEN object_type_attribute_id = 849 THEN c.text_value END) AS group_job
      , max(CASE WHEN object_type_attribute_id IN (5730) 
                 THEN regexp_extract(c.text_value, '^uid=([^,]+)',1) END)  AS uid_manager
    FROM delta.tech_data_platform_core.jira_ao_8542f1_ifj_obj ep
    JOIN delta.tech_data_platform_core.jira_ao_8542f1_ifj_obj_attr attr
      ON attr.object_id = ep.id
    JOIN delta.tech_data_platform_core.jira_ao_8542f1_ifj_obj_attr_val c 
      ON c.object_attribute_id = attr.id
    LEFT JOIN delta.tech_data_platform_core.jira_ao_8542f1_ifj_obj ref_obj
      ON c.referenced_object_id = ref_obj.id
    WHERE regexp_like(trim(lower(SPLIT_PART(ep.label, '@', 1))), '^[a-z]+[.][a-z]+$')
      AND ep.label NOT LIKE '%/%'
    GROUP BY 1
)
"""
cursor.execute(sql_users)
users_df = pd.DataFrame(cursor.fetchall(), columns=[c[0] for c in cursor.description])
users_df = users_df.astype("string")
users_df["uid"] = users_df["uid"].map(normalize_uid)

# 3) Explode users and LEFT JOIN to directory
exploded = (
    tbl[["base_table","users_list"]]
    .explode("users_list")
    .rename(columns={"users_list":"uid"})
)
exploded["uid"] = exploded["uid"].map(normalize_uid)

enriched = exploded.merge(users_df, on="uid", how="left")

# 4) Aggregate unique org attributes per table
def _uniq_json(series):
    vals = sorted({str(x).strip() for x in series if pd.notna(x) and str(x).strip()})
    return json.dumps(vals, ensure_ascii=False)

agg = (enriched.groupby("base_table", as_index=False)
       .agg(
           users_with_meta=("uid", lambda s: json.dumps(sorted({u for u in s if pd.notna(u)}), ensure_ascii=False)),
           jobs=("job", _uniq_json),
           departments=("department", _uniq_json),
           directions=("direction", _uniq_json),
           groups=("group_job", _uniq_json),
           team_leads=("uid_manager", _uniq_json),
       ))

# 5) Merge back and fill missing JSON arrays with "[]"
out = tbl.merge(agg, on="base_table", how="left")
for c in ["users_with_meta","jobs","departments","directions","groups","team_leads"]:
    if c in out.columns:
        out[c] = out[c].fillna("[]")

# Debug stats (optional): how many user rows matched attributes?
matched = (enriched["job"].notna() | enriched["department"].notna() | enriched["direction"].notna()).sum()
total_uids = len(enriched)
print(f"[diag] user rows with any attribute matched: {matched}/{total_uids} ({matched/total_uids:.1%})")

# 6) Save
out.to_csv(OUTPUT_ENRICHED_CSV, index=False)
print(f"[done] Saved: {OUTPUT_ENRICHED_CSV} (rows={len(out)})")

# Quick peek
with pd.option_context("display.max_colwidth", 140):
    display(out.head(10)[[
        "base_table","users_list","users_with_meta",
        "jobs","departments","directions","groups","team_leads"
    ]])


# Stage 3. Get data from DBT manifest

In [ ]:
# === dbt manifest → catalog + lineage (with downstream names, keys, partition_by, columns) ===
import json, re, sys
from pathlib import Path
from collections import defaultdict

import pandas as pd

# ======= CONFIG =======
MANIFEST_PATH = Path("dbt-dbt_datalake_manifest.json")   # download from here: https://nl-datalake-s3.prod.env:9001/browser/tech-data-platform-raw/dbt-dbt_datalake%2F
OUT_MODELS_CSV = Path("3_dbt_models_catalog.csv")
OUT_EDGES_CSV  = Path("3_dbt_lineage_edges.csv")

# ======= helpers =======
def _strip_quotes(s: str) -> str:
    if not isinstance(s, str):
        return ""
    return s.strip().strip('"').strip("'")

RELATION_RE = re.compile(r'^\s*"?(?P<db>[^"]+)"?\."?(?P<schema>[^"]+)"?\."?(?P<table>[^"]+)"?\s*$')

def parse_relation_name(relation_name: str) -> tuple[str|None, str|None, str|None]:
    """
    dbt often stores relation_name like "\"delta\".\"partnership_mart\".\"appsflyer_enriched_metrics_agg\""
    Returns (catalog, schema, table) or (None, None, None) if unparseable.
    """
    if not relation_name:
        return None, None, None
    m = RELATION_RE.match(relation_name)
    if not m:
        cleaned = relation_name.replace("\\", "")
        m = RELATION_RE.match(cleaned)
    if m:
        return _strip_quotes(m.group("db")), _strip_quotes(m.group("schema")), _strip_quotes(m.group("table"))
    return None, None, None

def base_table(schema: str|None, table: str|None) -> str|None:
    if schema and table:
        return f"{str(schema).lower()}.{str(table).lower()}"
    return None

def from_unrendered_or_config(node: dict, key: str, default=None):
    # Prefer unrendered_config[key] (more faithful), otherwise config[key]
    return (
        (node.get("unrendered_config") or {}).get(key)
        or (node.get("config") or {}).get(key)
        or default
    )

def _json(obj) -> str:
    try:
        return json.dumps(obj, ensure_ascii=False)
    except Exception:
        return "[]"

# ======= load manifest =======
if not MANIFEST_PATH.exists():
    sys.exit(f"File not found: {MANIFEST_PATH}")

with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

nodes: dict = manifest.get("nodes", {}) or {}
sources: dict = manifest.get("sources", {}) or {}

# ------- identities -------
def node_identity(n: dict) -> dict:
    cat, sch, tbl = parse_relation_name(n.get("relation_name", "") or "")
    if not sch or not tbl:
        sch = from_unrendered_or_config(n, "schema", n.get("schema"))
        tbl = from_unrendered_or_config(n, "alias", None) or n.get("alias") or n.get("name")
    cat = cat or from_unrendered_or_config(n, "database", n.get("database"))
    bt = base_table(sch, tbl)
    return {
        "catalog": cat,
        "schema": sch,
        "table": tbl,
        "base_table": bt,
        "fqtn": f"{cat}.{sch}.{tbl}" if (cat and sch and tbl) else None
    }

def source_identity(s: dict) -> dict:
    cat, sch, tbl = parse_relation_name(s.get("relation_name", "") or "")
    if not sch or not tbl:
        sch = s.get("schema", sch)
        tbl = s.get("name", tbl)
    cat = cat or s.get("database")
    bt = base_table(sch, tbl)
    return {
        "catalog": cat,
        "schema": sch,
        "table": tbl,
        "base_table": bt,
        "fqtn": f"{cat}.{sch}.{tbl}" if (cat and sch and tbl) else None
    }

node_id_to_ident = {uid: node_identity(n) for uid, n in nodes.items()}
source_id_to_ident = {uid: source_identity(s) for uid, s in sources.items()}

# ======= Build lineage edges and per-model rows =======
rows = []
edges = []  # (from_uid, to_uid, from_type, to_type, from_base_table, to_base_table)

for uid, n in nodes.items():
    if n.get("resource_type") != "model":
        continue

    ident = node_id_to_ident[uid]
    sch = ident["schema"]; tbl = ident["table"]; cat = ident["catalog"]; bt = ident["base_table"]

    # core props
    description  = n.get("description", "") or ""
    materialized = from_unrendered_or_config(n, "materialized", (n.get("config") or {}).get("materialized"))
    file_format  = from_unrendered_or_config(n, "file_format")
    schedule     = from_unrendered_or_config(n, "schedule")
    tags         = n.get("tags", []) or (n.get("config") or {}).get("tags", []) or []

    # NEW: unique_key & partition_by (as stored in config/unrendered_config)
    unique_key   = from_unrendered_or_config(n, "unique_key")
    partition_by = from_unrendered_or_config(n, "partition_by")

    # NEW: columns (names + detailed metadata from manifest)
    cols_block = (n.get("columns") or {})
    # columns list (ordered by name for stability)
    column_names = sorted([c_name for c_name in cols_block.keys()])
    # detailed structure: {name: {description, data_type, tags}}
    columns_detail = {}
    for c_name, c_meta in cols_block.items():
        if not isinstance(c_meta, dict):
            continue
        columns_detail[c_name] = {
            "description": c_meta.get("description") or "",
            "data_type": c_meta.get("data_type") or c_meta.get("type") or "",
            "tags": c_meta.get("tags") or []
        }

    # depends_on
    dep_node_ids = ((n.get("depends_on") or {}).get("nodes") or [])
    upstream_models = []
    upstream_sources = []
    resolved_upstream_tables = []

    for dep_id in dep_node_ids:
        if dep_id.startswith("model."):
            upstream_models.append(dep_id)
            from_bt = node_id_to_ident.get(dep_id, {}).get("base_table")
            if from_bt:
                resolved_upstream_tables.append(from_bt)
            edges.append((dep_id, uid, "model", "model",
                          node_id_to_ident.get(dep_id, {}).get("base_table"),
                          bt))
        elif dep_id.startswith("source."):
            upstream_sources.append(dep_id)
            from_bt = source_id_to_ident.get(dep_id, {}).get("base_table")
            if from_bt:
                resolved_upstream_tables.append(from_bt)
            edges.append((dep_id, uid, "source", "model",
                          source_id_to_ident.get(dep_id, {}).get("base_table"),
                          bt))
        else:
            # tests/macros/etc.
            pass

    # unique list & stable order
    def _uniq(seq):
        seen = set(); out = []
        for x in seq:
            if x not in seen:
                seen.add(x); out.append(x)
        return out

    upstream_models = _uniq(upstream_models)
    upstream_sources = _uniq(upstream_sources)
    resolved_upstream_tables = _uniq([x for x in resolved_upstream_tables if x])

    rows.append({
        "unique_id": uid,
        "catalog": cat,
        "schema": sch,
        "table": tbl,
        "base_table": bt,
        "fqtn": ident["fqtn"],
        "materialized": materialized,
        "file_format": file_format,
        "schedule": schedule,
        "tags": tags,
        "description": description,
        "unique_key": unique_key,
        "partition_by": partition_by,
        "columns": column_names,                # list of names
        "columns_detail": columns_detail,       # dict name -> {description, data_type, tags}
        "depends_on_models": upstream_models,
        "depends_on_sources": upstream_sources,
        "resolved_upstream_tables": resolved_upstream_tables,
    })

# ======= Compute reverse dependencies (depended_by) =======
children_by_parent = defaultdict(list)
for src_uid, dst_uid, src_type, dst_type, _, _ in edges:
    children_by_parent[src_uid].append(dst_uid)

# Attach downstream (as unique_ids and as names/FQTN) to each row
for r in rows:
    uid = r["unique_id"]
    child_uids = [c for c in children_by_parent.get(uid, []) if c.startswith("model.")]
    r["depended_by_models"] = child_uids

    # NEW: downstream model names/FQTN
    child_bt   = []
    child_fqtn = []
    for cu in child_uids:
        ident = node_id_to_ident.get(cu, {})
        if ident:
            if ident.get("base_table"):
                child_bt.append(ident["base_table"])
            if ident.get("fqtn"):
                child_fqtn.append(ident["fqtn"])
    # stable & unique
    r["depended_by_base_tables"] = sorted(set([x for x in child_bt if x]))
    r["depended_by_fqtn"]        = sorted(set([x for x in child_fqtn if x]))

# ======= Save outputs =======
df_models = pd.DataFrame(rows).sort_values(["schema","table"], na_position="last").reset_index(drop=True)
df_edges = pd.DataFrame(edges, columns=["from_uid","to_uid","from_type","to_type","from_base_table","to_base_table"])

# JSON-stringify complex columns for CSV
for col in ["tags", "columns", "columns_detail",
            "depends_on_models", "depends_on_sources", "resolved_upstream_tables",
            "depended_by_models", "depended_by_base_tables", "depended_by_fqtn"]:
    if col in df_models.columns:
        df_models[col] = df_models[col].apply(_json)

df_models.to_csv(OUT_MODELS_CSV, index=False)
df_edges.to_csv(OUT_EDGES_CSV, index=False)

print(f"[ok] Wrote {OUT_MODELS_CSV} with {len(df_models)} models")
print(f"[ok] Wrote {OUT_EDGES_CSV} with {len(df_edges)} lineage edges")

# (optional) quick peek
with pd.option_context("display.max_colwidth", 120):
    display(df_models.head(5)[[
        "schema","table","materialized","file_format","schedule",
        "unique_key","partition_by","columns","depended_by_base_tables"
    ]])


# Stage 4. DataHub: check whether the asset is available in DH and get URL

In [ ]:
#new
# === Check tables in DataHub and enrich with URL, descriptions, tags (delta via s3/minio; iceberg skipped) ===
import os, sys, json, time, logging, tempfile, shutil
from pathlib import Path
from urllib.parse import quote
import pandas as pd
import requests

# ------------- CONFIG -------------
# INPUT_CSV   = "master_tables.csv"                 # must have: fqtn, catalog, schema, table_name

INPUT_TABLE = "delta.tech_data_platform_work.astarostina_master_table_metadata"
OUTPUT_CSV  = "4_master_with_datahub.csv"

DATAHUB_API_URL = "https://data-catalog.exness.io/api/graphql"
DATAHUB_UI_BASE = "https://data-catalog.exness.io"  # UI host

# Bearer token
DATAHUB_TOKEN = credscram.creds.datahub_token  # or: os.environ["DATAHUB_TOKEN"]

# Platform/name mapping for URNs:
#   delta -> s3 platform, dataset name = minio.<schema>.<table>
#   iceberg -> NOT AVAILABLE in DataHub today (skip)
PLATFORM_MAP = {
    "delta":   ("s3", lambda sch, tbl: f"minio.{sch}.{tbl}"),
    # "iceberg": (None, None)  # explicitly unsupported
}

CHECKPOINT_DIR = Path("checkpoints")
CKPT_FILE      = CHECKPOINT_DIR / "ckpt_datahub_master_20251229.csv"

RETRY_ATTEMPTS = 3
RETRY_SLEEP_S  = 1.5
REQ_TIMEOUT_S  = 20

# ------------- Logging -------------
CHECKPOINT_DIR.mkdir(exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
log = logging.getLogger("datahub-master")

# ------------- Helpers -------------
def _atomic_write_csv(df: pd.DataFrame, path: Path):
    fd, tmp = tempfile.mkstemp(prefix="tmp_", dir=str(CHECKPOINT_DIR), text=True); os.close(fd)
    df.to_csv(tmp, index=False)
    shutil.move(tmp, str(path))

def _append_row_csv(row: dict, path: Path, header_fields: list[str]):
    exists = path.exists() and path.stat().st_size > 0
    df = pd.DataFrame([row], columns=header_fields)
    if exists:
        with open(path, "a", encoding="utf-8") as f:
            df.to_csv(f, index=False, header=False)
    else:
        df.to_csv(path, index=False)

def _json(obj):
    try:
        return json.dumps(obj, ensure_ascii=False)
    except Exception:
        return "[]"

def _graph_payload(urn: str) -> dict:
    # Pull properties, editableProperties, and tags
    q = """
    query getDs($urn: String!) {
      dataset(urn: $urn) {
        properties { name description }
        editableProperties { description }
        globalTags {
          tags { tag { urn name } }
        }
        platform { name }
        urn
      }
    }
    """
    return {"query": q, "variables": {"urn": urn}}

def _do_request(payload: dict) -> dict | None:
    headers = {
        "Authorization": f"Bearer {DATAHUB_TOKEN}",
        "Content-Type": "application/json",
    }
    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            r = requests.post(DATAHUB_API_URL, headers=headers, json=payload, timeout=REQ_TIMEOUT_S)
            if r.status_code == 200:
                return r.json()
            if r.status_code in (429, 500, 502, 503, 504):
                log.warning(f"[datahub] transient {r.status_code} attempt {attempt}/{RETRY_ATTEMPTS}")
                time.sleep(RETRY_SLEEP_S * attempt)
                continue
            log.error(f"[datahub] HTTP {r.status_code}: {r.text[:400]}")
            return None
        except Exception as e:
            log.warning(f"[datahub] error {e} attempt {attempt}/{RETRY_ATTEMPTS}")
            time.sleep(RETRY_SLEEP_S * attempt)
    return None

def _urn_from_row(catalog: str, schema: str, table: str) -> tuple[str|None, str|None]:
    """
    Returns (dataset_name, URN) for supported catalogs.
    - delta -> s3/minio.<schema>.<table>
    - iceberg -> None (skipped)
    """
    if not catalog or not schema or not table:
        return None, None
    cat = str(catalog).strip().lower()
    sch = str(schema).strip().lower()
    tbl = str(table).strip().lower()

    if cat not in PLATFORM_MAP:
        return None, None

    platform, name_fn = PLATFORM_MAP[cat]
    if not platform or not name_fn:
        return None, None  # unsupported

    dataset_name = name_fn(sch, tbl)
    urn = f"urn:li:dataset:(urn:li:dataPlatform:{platform},{dataset_name},PROD)"
    return dataset_name, urn

def _ui_url_for_urn(urn: str) -> str:
    # DataHub UI deep link for dataset pages is /dataset/<encoded URN>
    return f"{DATAHUB_UI_BASE}/dataset/{quote(urn, safe='')}"

# ------------- 1) Load base -------------

# assert os.path.exists(INPUT_CSV), f"File not found: {INPUT_CSV}"
# base = pd.read_csv(INPUT_CSV, low_memory=False)
base = pd.read_sql_query(f"""
    SELECT
        catalog || '.'|| schema || '.'|| table_name as fqtn,
        catalog,
        schema,
        table_name
    FROM {INPUT_TABLE}
""", conn)

log.info(f"[input] Loaded {len(base)} rows from {INPUT_CSV}")

need_cols = {"fqtn", "catalog", "schema", "table_name"}
missing = need_cols - set(base.columns)
if missing:
    raise ValueError(f"Missing columns in input: {missing}")

# ------------- 2) Build tasks -------------
tasks = []
for _, r in base.iterrows():
    cat = r.get("catalog")
    sch = r.get("schema")
    tbl = r.get("table_name")
    if pd.isna(cat) or pd.isna(sch) or pd.isna(tbl):
        continue
    dataset_name, urn = _urn_from_row(cat, sch, tbl)
    if urn:
        tasks.append({"fqtn": r["fqtn"], "catalog": str(cat).lower(), "dataset_name": dataset_name, "urn": urn})

tasks_df = pd.DataFrame(tasks).drop_duplicates(subset=["fqtn"])
log.info(f"[build] tasks to fetch: {len(tasks_df)}")

# If nothing to fetch, just add empty columns and save (but flag iceberg rows as unavailable)
if tasks_df.empty:
    out = base.copy()
    out["dh_found"] = False
    out["dh_urn"]   = ""
    out["dh_url"]   = ""
    out["dh_description"] = ""
    out["dh_editable_description"] = ""
    out["dh_tags"]  = "[]"
    # For iceberg rows, add a helper flag so you can see why it's empty
    out["dh_reason"] = out["catalog"].str.lower().eq("iceberg").map({True: "not_available_in_datahub", False: ""})
    out.to_csv(OUTPUT_CSV, index=False)
    log.info(f"[done] Saved {OUTPUT_CSV} (no delta tasks to query)")
else:
    # ------------- 3) Checkpoint init/load -------------
    ckpt_cols = ["fqtn", "urn", "status", "description", "editable_description", "tags_json", "url"]
    if not CKPT_FILE.exists():
        pd.DataFrame(columns=ckpt_cols).to_csv(CKPT_FILE, index=False)

    ckpt = pd.read_csv(CKPT_FILE) if CKPT_FILE.stat().st_size > 0 else pd.DataFrame(columns=ckpt_cols)
    done = set(ckpt["fqtn"]) if not ckpt.empty else set()

    # Also materialize iceberg "unavailable" into checkpoint (one time)
    iceberg_fqtns = base.loc[base["catalog"].str.lower().eq("iceberg"), "fqtn"].dropna().unique().tolist()
    iceberg_pending = [fq for fq in iceberg_fqtns if fq not in done]
    if iceberg_pending:
        for fq in iceberg_pending:
            _append_row_csv({
                "fqtn": fq,
                "urn": "",
                "status": "unavailable",
                "description": "",
                "editable_description": "",
                "tags_json": "[]",
                "url": ""
            }, CKPT_FILE, ckpt_cols)
        ckpt = pd.read_csv(CKPT_FILE) if CKPT_FILE.stat().st_size > 0 else pd.DataFrame(columns=ckpt_cols)
        done = set(ckpt["fqtn"]) if not ckpt.empty else set()

    todo_df = tasks_df[~tasks_df["fqtn"].isin(done)].reset_index(drop=True)
    log.info(f"[datahub] to fetch now: {len(todo_df)} (already in ckpt: {len(done)})")

    # ------------- 4) Query loop with per-row checkpoint -------------
    cnt = 0
    for _, r in todo_df.iterrows():
        urn = r["urn"]
        payload = _graph_payload(urn)
        js = _do_request(payload)

        status = "not_found"
        desc = ""
        edesc = ""
        tags = []

        if js and "data" in js and js["data"].get("dataset"):
            ds = js["data"]["dataset"]
            # properties.description
            props = (ds.get("properties") or {})
            desc = props.get("description") or ""

            # editableProperties.description
            eprops = (ds.get("editableProperties") or {})
            edesc = eprops.get("description") or ""

            # tags: globalTags.tags[].tag.name
            gtags = (ds.get("globalTags") or {}).get("tags") or []
            for t in gtags:
                tag = (t or {}).get("tag") or {}
                nm = tag.get("name") or ""
                if nm:
                    tags.append(nm)

            status = "ok"

        _append_row_csv({
            "fqtn": r["fqtn"],
            "urn": urn,
            "status": status,
            "description": desc,
            "editable_description": edesc,
            "tags_json": _json(sorted(set(tags))),
            "url": _ui_url_for_urn(urn)
        }, CKPT_FILE, ckpt_cols)

        cnt += 1
        if cnt % 25 == 0:
            log.info(f"[datahub] progress {cnt}/{len(todo_df)}")

    # ------------- 5) Merge back and save -------------
    dh = pd.read_csv(CKPT_FILE, low_memory=False)

    enr = (
        dh.rename(columns={
            "urn": "dh_urn",
            "url": "dh_url",
            "description": "dh_description",
            "editable_description": "dh_editable_description",
            "tags_json": "dh_tags"
        })
        .assign(dh_found=lambda d: d["status"].eq("ok"))
    )

    out = base.merge(
        enr[["fqtn", "dh_found", "dh_urn", "dh_url", "dh_description",
             "dh_editable_description", "dh_tags", "status"]],
        on="fqtn", how="left"
    )

    # Helpful reason column: iceberg unavailable, or missing
    out["dh_reason"] = ""
    out.loc[out["catalog"].str.lower().eq("iceberg"), "dh_reason"] = "not_available_in_datahub"
    out.loc[out["status"].eq("not_found"), "dh_reason"] = "not_found"
    out.loc[out["status"].eq("unavailable"), "dh_reason"] = "not_available_in_datahub"

    out.drop(columns=["status"], inplace=True, errors="ignore")

    # Fill NaNs for nice CSV
    fills = {
        "dh_found": False, "dh_urn": "", "dh_url": "",
        "dh_description": "", "dh_editable_description": "", "dh_tags": "[]",
        "dh_reason": ""
    }
    for c, v in fills.items():
        if c not in out.columns:
            out[c] = v
        else:
            out[c] = out[c].fillna(v)

    out.to_csv(OUTPUT_CSV, index=False)
    log.info(f"[done] Saved {OUTPUT_CSV} with {len(out)} rows; checkpoint: {CKPT_FILE}")
    with pd.option_context("display.max_colwidth", 100):
        display(out.head(10)[["fqtn","catalog","dh_found","dh_url","dh_reason","dh_description","dh_editable_description","dh_tags"]])


# Stage 5. DataHub: Get desc of upstream tables from from DH (depends on parsed file of dbt manifest - Stage 3)

In [ ]:
# optional - in case of credscram is not set yet
import credscram
from getpass import getpass

credscram.creds.set("datahub_token",getpass("Datahub token: "))

In [ ]:
# === DataHub enrichment by resolved_upstream_tables via name-only search ===
import os, sys, json, time, logging, tempfile, shutil
from pathlib import Path
import pandas as pd
import requests

# ---------------- CONFIG ----------------
DATAHUB_URL   = "https://data-catalog.exness.io/api/graphql"
DATAHUB_TOKEN = credscram.creds.datahub_token

INPUT_CSV   = "3_dbt_models_catalog.csv"      # fqtn, origin, resolved_upstream_tables
OUTPUT_CSV  = "5_dbt_enriched_with_datahub_sources_desc.csv"

CHECKPOINT_DIR = Path("checkpoints")
CKPT_FILE      = CHECKPOINT_DIR / "ckpt_datahub_upstreams_20251229.csv"

RETRY_ATTEMPTS = 3
RETRY_SLEEP_S  = 2.0
REQ_TIMEOUT_S  = 20

# ---------------- Logging ----------------
CHECKPOINT_DIR.mkdir(exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
log = logging.getLogger("datahub-upstreams")

# ---------------- Helpers ----------------
def _json(obj):
    try:
        return json.dumps(obj, ensure_ascii=False)
    except Exception:
        return "[]"

def _atomic_write_csv(df: pd.DataFrame, path: Path):
    fd, tmp = tempfile.mkstemp(prefix="tmp_", dir=str(CHECKPOINT_DIR), text=True); os.close(fd)
    df.to_csv(tmp, index=False)
    shutil.move(tmp, str(path))

def _append_row_csv(row: dict, path: Path, header_fields: list[str]):
    exists = path.exists() and path.stat().st_size > 0
    df = pd.DataFrame([row], columns=header_fields)
    if exists:
        with open(path, "a", encoding="utf-8") as f:
            df.to_csv(f, index=False, header=False)
    else:
        df.to_csv(path, index=False)

def _table_name_from_upstream(upstream: str) -> str:
    """
    Accepts forms like 'schema.table', 'db.schema.table', or 'table'.
    Returns just the last token (table name only).
    """
    if not isinstance(upstream, str) or not upstream.strip():
        return ""
    parts = [p for p in upstream.strip().split(".") if p]
    return parts[-1].lower() if parts else ""

def _graph_search_by_name_payload(table_name: str, start: int = 0, count: int = 3) -> dict:
    """
    Single-call GraphQL search that returns dataset fields + tags.
    We search with 'query' equal to **table_name only**.
    """
    q = """
    query searchDatasets($input: SearchInput!) {
      search(input: $input) {
        start
        count
        total
        searchResults {
          entity {
            urn
            ... on Dataset {
              properties { name description }
              platform { name }
              tags { tags { tag { name } } }
              ownership { owners { owner { ... on CorpUser { urn } } } }
              editableProperties { description }
            }
          }
        }
      }
    }
    """
    variables = {
        "input": {
            "type": "DATASET",
            "query": table_name,   # <— name-only search
            "start": start,
            "count": count
        }
    }
    return {"query": q, "variables": variables}

def _do_request(payload: dict) -> dict | None:
    headers = {
        "Authorization": f"Bearer {DATAHUB_TOKEN}",
        "Content-Type": "application/json",
    }
    for attempt in range(1, RETRY_ATTEMPTS+1):
        try:
            r = requests.post(DATAHUB_URL, headers=headers, json=payload, timeout=REQ_TIMEOUT_S)
            if r.status_code == 200:
                return r.json()
            if r.status_code in (429, 502, 503, 504):
                log.warning(f"[datahub] transient {r.status_code} attempt {attempt}/{RETRY_ATTEMPTS}")
                time.sleep(RETRY_SLEEP_S * attempt)
                continue
            log.error(f"[datahub] HTTP {r.status_code}: {r.text[:400]}")
            return None
        except Exception as e:
            log.warning(f"[datahub] error {e} attempt {attempt}/{RETRY_ATTEMPTS}")
            time.sleep(RETRY_SLEEP_S * attempt)
    return None

def _parse_upstream_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            cand = json.loads(x)
            if isinstance(cand, list):
                return cand
        except Exception:
            s = x.strip().strip("[]")
            if not s:
                return []
            parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
            return [p for p in parts if p]
    return []

# ---------------- 1) Load base ----------------
if "final_df" in globals() and isinstance(final_df, pd.DataFrame):
    base = final_df.copy()
    log.info(f"[input] Using in-memory final_df with {len(base)} rows")
else:
    assert os.path.exists(INPUT_CSV), f"File not found: {INPUT_CSV}"
    base = pd.read_csv(INPUT_CSV, low_memory=False)
    log.info(f"[input] Loaded {len(base)} rows from {INPUT_CSV}")

need = {"fqtn", "resolved_upstream_tables"}
missing = need - set(base.columns)
if missing:
    raise ValueError(f"Missing columns in input: {missing}")

# ---------------- 2) Build tasks (fqtn, upstream, search_name) ----------------
tasks = []
for _, row in base.iterrows():
    ups = _parse_upstream_list(row.get("resolved_upstream_tables"))
    if not ups:
        continue
    for u in ups:
        tn = _table_name_from_upstream(u)  # name-only
        if not tn:
            continue
        tasks.append({"fqtn": row["fqtn"], "upstream": u, "search_name": tn})

tasks_df = pd.DataFrame(tasks)
log.info(f"[build] tasks to fetch: {len(tasks_df)}")

if tasks_df.empty:
    base.to_csv(OUTPUT_CSV, index=False)
    log.info(f"[done] Saved {OUTPUT_CSV} (no upstream tasks)")
else:
    # ---------------- 3) Checkpoint init/load ----------------
    ckpt_cols = [
        "fqtn", "upstream", "search_name", "status",
        "dataset_urn", "dataset_name", "platform",
        "description", "editable_description", "owners_json",
        "tags_json"  # <-- NEW: array of tag names only
    ]
    if not CKPT_FILE.exists():
        pd.DataFrame(columns=ckpt_cols).to_csv(CKPT_FILE, index=False)
    ckpt = pd.read_csv(CKPT_FILE) if CKPT_FILE.stat().st_size > 0 else pd.DataFrame(columns=ckpt_cols)

    done_keys: set[tuple[str, str]] = set()
    if not ckpt.empty:
        ckpt["fqtn"] = ckpt["fqtn"].astype(str)
        ckpt["upstream"] = ckpt["upstream"].astype(str)
        done_keys = set(zip(ckpt["fqtn"], ckpt["upstream"]))

    # ---------------- 4) TODO set ----------------
    tasks_df = tasks_df.dropna(subset=["upstream"])
    tasks_df = tasks_df[tasks_df["upstream"].astype(str).str.len() > 0]
    tasks_df = tasks_df.drop_duplicates(subset=["fqtn", "upstream"])

    mask_todo = [ (fqtn, up) not in done_keys for fqtn, up in zip(tasks_df["fqtn"].astype(str),
                                                                  tasks_df["upstream"].astype(str)) ]
    todo_df = tasks_df.loc[mask_todo].reset_index(drop=True)
    log.info(f"[datahub] to fetch now: {len(todo_df)} (already in ckpt: {len(done_keys)})")

    # ---------------- 5) Query loop with checkpoint per row ----------------
    cnt = 0
    for _, r in todo_df.iterrows():
        js = _do_request(_graph_search_by_name_payload(r["search_name"], start=0, count=3))

        status = "not_found"
        dataset_urn = ""
        dataset_name = ""
        platform = ""
        desc = ""
        editable_desc = ""
        owners = []
        tag_names = []

        try:
            sr = (((js or {}).get("data") or {}).get("search") or {}).get("searchResults") or []
            # Take the first dataset hit, if any
            if sr:
                ent = (sr[0] or {}).get("entity") or {}
                dataset_urn = ent.get("urn") or ""
                ds = ent  # already ... on Dataset
                props = (ds.get("properties") or {})
                dataset_name = (props.get("name") or "").strip()
                desc = props.get("description") or ""
                platform = ((ds.get("platform") or {}).get("name") or "").strip()

                # owners
                own = (ds.get("ownership") or {}).get("owners") or []
                for o in own:
                    ou = (((o or {}).get("owner") or {}).get("urn") or "")
                    if ou:
                        owners.append(ou.split(":")[-1])

                # editable desc
                editable_props = (ds.get("editableProperties") or {})
                if isinstance(editable_props, dict):
                    editable_desc = editable_props.get("description") or ""

                # tags -> array of names only
                tags = ((ds.get("tags") or {}).get("tags") or [])
                tag_names = [ (t or {}).get("tag", {}).get("name") for t in tags ]
                tag_names = [t for t in tag_names if isinstance(t, str) and t]

                status = "ok"
        except Exception as e:
            log.warning(f"[datahub] parse error for {r['search_name']}: {e}")

        _append_row_csv({
            "fqtn": r["fqtn"],
            "upstream": r["upstream"],
            "search_name": r["search_name"],
            "status": status,
            "dataset_urn": dataset_urn,
            "dataset_name": dataset_name,
            "platform": platform,
            "description": desc,
            "editable_description": editable_desc,
            "owners_json": _json(owners),
            "tags_json": _json(tag_names),  # <-- ONLY names e.g. ["vertica:daily","vertica:partnership"]
        }, CKPT_FILE, ckpt_cols)

        cnt += 1
        if cnt % 25 == 0:
            log.info(f"[datahub] progress {cnt}/{len(todo_df)}")

    # reread checkpoint
    dh = pd.read_csv(CKPT_FILE, low_memory=False)

# ---------------- 6) Aggregate per fqtn and merge back (robust, no FutureWarning) ----------------
def _safe_list(s):
    if isinstance(s, str) and s.startswith("["):
        try:
            x = json.loads(s)
            return x if isinstance(x, list) else []
        except Exception:
            return []
    return []

rows = []
for fqtn, g in dh.groupby("fqtn", dropna=False):
    # compute all aggregates, even if empty -> ensure columns exist
    matches = _json(sorted([
        u for u, s in zip(g["upstream"], g["status"])
        if isinstance(s, str) and s == "ok" and isinstance(u, str) and u
    ]))

    descriptions = {
        u: d for u, d, s in zip(g["upstream"], g["description"], g["status"])
        if isinstance(s, str) and s == "ok" and isinstance(u, str) and u and isinstance(d, str) and d.strip()
    }
    editable_descriptions = {
        u: ed for u, ed, s in zip(g["upstream"], g["editable_description"], g["status"])
        if isinstance(s, str) and s == "ok" and isinstance(u, str) and u and isinstance(ed, str) and ed.strip()
    }
    owners = {
        u: _safe_list(oj)
        for u, oj, s in zip(g["upstream"], g["owners_json"], g["status"])
        if isinstance(s, str) and s == "ok" and isinstance(u, str) and u
    }
    tags_map = {
        u: _safe_list(tj)
        for u, tj, s in zip(g["upstream"], g["tags_json"], g["status"])
        if isinstance(s, str) and s == "ok" and isinstance(u, str) and u
    }

    rows.append({
        "fqtn": fqtn,
        "matches": _json(matches if isinstance(matches, list) else json.loads(matches)),
        "descriptions": _json(descriptions),
        "editable_descriptions": _json(editable_descriptions),
        "owners": _json(owners),
        "tags": _json(tags_map),
    })

agg = pd.DataFrame(rows)

out = base.merge(agg, on="fqtn", how="left")

# Ensure all expected columns exist (avoid KeyError on selection)
for col, default in [
    ("matches", "[]"),
    ("descriptions", "{}"),
    ("editable_descriptions", "{}"),
    ("owners", "{}"),
    ("tags", "{}"),
]:
    if col not in out.columns:
        out[col] = default
    else:
        out[col] = out[col].fillna(default)

# Take only requested columns and rename to final names
wanted_cols = [
    "fqtn", "resolved_upstream_tables",
    "matches", "descriptions", "editable_descriptions", "owners", "tags"
]
out_trim = out[wanted_cols].rename(columns={
    "descriptions": "upstream_descriptions",
    "editable_descriptions": "upstream_editable_descs",
    "owners": "upstream_owners",
    "tags": "upstream_tags",
})

_atomic_write_csv(out_trim, Path(OUTPUT_CSV))
log.info(f"[done] Saved {OUTPUT_CSV} with {len(out_trim)} rows; checkpoint: {CKPT_FILE}")

with pd.option_context("display.max_colwidth", 120):
    display(out_trim.head(10))


# Stage 6. Merge outputs from all stages

In [ ]:
#### Updated 20251229
# === Union + Merge (robust normalization; correct usage join; diagnostics) ===
import ast, json, re
import pandas as pd
from pathlib import Path

MASTER_TABLE_METADATA_SQL = f"""
SELECT
    t.catalog,
    t.schema,
    t.table_type,
    t.table_name,
    columns,
    partitioned_by,
    last_data_update as etl_updated_at

FROM delta.tech_data_platform_work.astarostina_master_table_metadata t
"""


# ---------- Paths ----------
#PATH_S1 = Path("master_tables.csv")                       # Stage 1 (must have fqtn OR catalog+schema+table_name)

PATH_S2 = Path("2_table_stats_final.csv")                 # Stage 2 (usage stats; must end with base_table schema.table)
PATH_S3 = Path("3_dbt_models_catalog.csv")                # Stage 3 (dbt; fqtn or database+schema+table)
PATH_DH_MASTER = Path("4_master_with_datahub.csv")          # Stage 5 (optional)
PATH_DH_UP     = Path("5_dbt_enriched_with_datahub_sources_desc.csv")  # Stage 6 (optional)

OUT_PATH = Path("6_tables_union_all.csv")

# ---------- Helpers ----------
def _safe_str(x): 
    return "" if x is None else str(x)

def _norm_fqtn(x: str) -> str:
    if not isinstance(x, str): return ""
    # remove quotes/backticks/brackets and compress spaces
    y = x.replace('"','').replace('`','').replace('[','').replace(']','')
    y = re.sub(r'\s+', ' ', y).strip().lower()
    return y

def _to_base_table_from_any(x: str) -> str | None:
    """
    Take any of:
      - schema.table
      - catalog.schema.table
      - possibly quoted/bracketed/cased
    Return: 'schema.table' (lower) or None
    """
    if not isinstance(x, str) or not x.strip():
        return None
    y = _norm_fqtn(x)
    parts = [p for p in y.split('.') if p]
    if len(parts) >= 2:
        s, t = parts[-2], parts[-1]
        return f"{s}.{t}"
    return None

def _make_fqtn(catalog: str, schema: str, table: str) -> str | None:
    if all([catalog, schema, table]):
        return f"{_safe_str(catalog).strip().lower()}." \
               f"{_safe_str(schema).strip().lower()}." \
               f"{_safe_str(table).strip().lower()}"
    return None

def _parse_listish(x):
    if isinstance(x, list): return x
    if isinstance(x, str):
        xs = x.strip()
        if not xs: return []
        try:
            return json.loads(xs)
        except Exception:
            try:
                return ast.literal_eval(xs)
            except Exception:
                return [p.strip().strip("'").strip('"') for p in xs.strip("[]").split(",") if p.strip()]
    return []

def _first_catalog(catalogs_value):
    lst = _parse_listish(catalogs_value)
    return (lst[0].strip().lower() if lst else None)

def _ensure_fqtn_and_base(df: pd.DataFrame, guess_cols: dict[str, str]) -> pd.DataFrame:
    """
    Ensure df has:
      - fqtn (normalized)
      - base_table (schema.table, normalized)
    We try in order:
      base_table from (schema, table/table_name),
      else from fqtn,
      else from any 'table' column via _to_base_table_from_any.
    """
    df = df.copy()

    # fqtn
    if "fqtn" not in df.columns or df["fqtn"].isna().all():
        cat_c = guess_cols.get("catalog")
        sch_c = guess_cols.get("schema")
        tbl_c = guess_cols.get("table")
        if sch_c and tbl_c:
            df["fqtn"] = df.apply(lambda r: _make_fqtn(_safe_str(r.get(cat_c,"")),
                                                       _safe_str(r.get(sch_c,"")),
                                                       _safe_str(r.get(tbl_c,""))), axis=1)
        else:
            df["fqtn"] = None
    df["fqtn"] = df["fqtn"].astype(str).map(_norm_fqtn)

    # base_table
    if "base_table" not in df.columns or df["base_table"].isna().all():
        # try from explicit schema/table columns
        sch_c = guess_cols.get("schema")
        tbl_c = guess_cols.get("table")
        if sch_c and tbl_c and sch_c in df.columns and tbl_c in df.columns:
            df["base_table"] = (df[sch_c].astype(str).map(_norm_fqtn) + "." +
                                df[tbl_c].astype(str).map(_norm_fqtn)).map(_to_base_table_from_any)
        else:
            # try from fqtn
            df["base_table"] = df["fqtn"].map(_to_base_table_from_any)

    # normalize any existing base_table
    df["base_table"] = df["base_table"].astype(str).map(_to_base_table_from_any)
    return df

def _prefix_except(df: pd.DataFrame, prefix: str, keep: list[str]) -> pd.DataFrame:
    rename_map = {c: f"{prefix}{c}" for c in df.columns if c not in keep}
    return df.rename(columns=rename_map)

# ---------- Load ----------
s1 = pd.read_sql_query(MASTER_TABLE_METADATA_SQL, conn)
s2 = pd.read_csv(PATH_S2, low_memory=False)
s3 = pd.read_csv(PATH_S3, low_memory=False)

dh_master = pd.read_csv(PATH_DH_MASTER, low_memory=False) if PATH_DH_MASTER.exists() else pd.DataFrame(columns=["fqtn"])
dh_up     = pd.read_csv(PATH_DH_UP,     low_memory=False) if PATH_DH_UP.exists()     else pd.DataFrame(columns=["fqtn"])

# ---------- Normalize keys (strong) ----------
# Stage 1 (master)
s1 = _ensure_fqtn_and_base(
    s1,
    guess_cols={
        "catalog": next((c for c in ["catalog","database"] if c in s1.columns), None),
        "schema":  next((c for c in ["schema"] if c in s1.columns), None),
        "table":   next((c for c in ["table_name","table","name"] if c in s1.columns), None),
    }
)

# Stage 2 (usage): force a normalized base_table no matter what the column looked like
if "base_table" not in s2.columns or s2["base_table"].isna().all():
    # try from fqtn
    if "fqtn" in s2.columns and not s2["fqtn"].isna().all():
        s2["base_table"] = s2["fqtn"].map(_to_base_table_from_any)
    # else try from 'table' shaped like schema.table or catalog.schema.table
    elif "table" in s2.columns:
        s2["base_table"] = s2["table"].map(_to_base_table_from_any)
    else:
        s2["base_table"] = None
# normalize base_table even if it already existed
s2["base_table"] = s2["base_table"].astype(str).map(_to_base_table_from_any)

# Stage 3 (dbt)
s3 = _ensure_fqtn_and_base(
    s3,
    guess_cols={
        "catalog": next((c for c in ["database","catalog"] if c in s3.columns), None),
        "schema":  next((c for c in ["schema"] if c in s3.columns), None),
        "table":   next((c for c in ["table","name"] if c in s3.columns), None),
    }
)

# DataHub files (normalize fqtn only; they keep master keys)
if not dh_master.empty:
    if "fqtn" not in dh_master.columns:
        cand = {c.lower(): c for c in dh_master.columns}
        cat_c = cand.get("catalog") or cand.get("database")
        sch_c = cand.get("schema")
        tbl_c = cand.get("table") or cand.get("table_name") or cand.get("name")
        if sch_c and tbl_c:
            dh_master["fqtn"] = dh_master.apply(lambda r: _make_fqtn(_safe_str(r.get(cat_c,"")),
                                                                      _safe_str(r.get(sch_c,"")),
                                                                      _safe_str(r.get(tbl_c,""))), axis=1)
        else:
            dh_master["fqtn"] = None
    dh_master["fqtn"] = dh_master["fqtn"].astype(str).map(_norm_fqtn)

if not dh_up.empty and "fqtn" in dh_up.columns:
    dh_up["fqtn"] = dh_up["fqtn"].astype(str).map(_norm_fqtn)

# ---------- Deduplicate dbt on fqtn ----------
if not s3.empty:
    s3 = s3.sort_values(s3.columns.tolist()).groupby("fqtn", as_index=False).first()

# ---------- Prefixes ----------
# ---------- Prefixes ----------
# Keep join keys unprefixed ONLY on the master side
s1_pref  = _prefix_except(s1,  "master_", keep=["fqtn","base_table"])
s2_pref  = _prefix_except(s2,  "usage_",  keep=["base_table"])

# IMPORTANT: do NOT keep base_table unprefixed on s3 — let it be prefixed
s3_pref  = _prefix_except(s3,  "dbt_",    keep=["fqtn"])   # base_table -> dbt_base_table

dhm_pref = _prefix_except(dh_master, "dh_",    keep=["fqtn"])
dhu_pref = _prefix_except(dh_up,     "dh_up_", keep=["fqtn"])

# ---------- Merge master ⟂ dbt on fqtn ----------
master = s1_pref.merge(s3_pref, on="fqtn", how="outer")

# Prefer a single unified base_table for downstream joins:
# use master.base_table if present, else dbt_base_table
if "base_table" not in master.columns:
    master["base_table"] = master.get("dbt_base_table")
else:
    master["base_table"] = master["base_table"].fillna(master.get("dbt_base_table"))

# normalize once more (safety)
master["base_table"] = master["base_table"].astype(str).map(_to_base_table_from_any)

# ---------- DIAGNOSTICS before usage join ----------
mb = set(master["base_table"].dropna())
ub = set(s2_pref["base_table"].dropna())
print(f"[diag] master base_tables: {len(mb)}; usage base_tables: {len(ub)}; intersection: {len(mb & ub)}")
if len(mb & ub) == 0:
    print("[diag] examples master:", list(sorted(mb))[:5])
    print("[diag] examples usage :", list(sorted(ub))[:5])

# ---------- Left-join usage on base_table ----------
merged = master.merge(s2_pref, on="base_table", how="left")

# ---------- Append usage-only rows that truly have no master/dbt match ----------
matched_bt = set(merged["base_table"].dropna())
s2_only_bt = sorted(ub - matched_bt)
print(f"[diag] usage-only base_tables after join: {len(s2_only_bt)}")

if s2_only_bt:
    s2_only = s2_pref[s2_pref["base_table"].isin(s2_only_bt)].copy()

    # try to synthesize fqtn for these rows using 'catalogs' from the original s2 (if present)
    s2_org = s2.copy()
    if "catalogs" in s2_org.columns:
        s2_org["_first_catalog"] = s2_org["catalogs"].map(_first_catalog)
    else:
        s2_org["_first_catalog"] = None
    s2_org["schema"] = s2_org["base_table"].map(lambda x: x.split(".",1)[0] if isinstance(x,str) and "." in x else None)
    s2_org["table"]  = s2_org["base_table"].map(lambda x: x.split(".",1)[1] if isinstance(x,str) and "." in x else None)
    s2_org["fqtn"]   = s2_org.apply(lambda r: _make_fqtn(r.get("_first_catalog"), r.get("schema"), r.get("table")), axis=1)
    s2_masterish = s2_org[["fqtn","base_table"]].drop_duplicates()

    s2_appended = s2_masterish.merge(s2_only, on="base_table", how="left")
    merged = pd.concat([merged, s2_appended], ignore_index=True, sort=False)


# ---------- DIAGNOSTICS before usage join ----------
mb = set(master["base_table"].dropna())
ub = set(s2_pref["base_table"].dropna())
print(f"[diag] master base_tables: {len(mb)}; usage base_tables: {len(ub)}; intersection: {len(mb & ub)}")

if len(mb & ub) == 0:
    # show a few examples to understand formatting
    print("[diag] examples master:", list(sorted(mb))[:5])
    print("[diag] examples usage :", list(sorted(ub))[:5])

# ---------- Left-join usage on base_table (now reliably normalized) ----------
merged = master.merge(s2_pref, on="base_table", how="left")

# ---------- Left-join DataHub enrichments ----------
if not dhm_pref.empty:
    merged = merged.merge(dhm_pref, on="fqtn", how="left")
if not dhu_pref.empty:
    merged = merged.merge(dhu_pref, on="fqtn", how="left")

# ---------- Final polish & save ----------
merged = merged.drop_duplicates(subset=["fqtn","base_table"], keep="first").reset_index(drop=True)
merged.to_csv(OUT_PATH, index=False)
print(f"[done] Saved {OUT_PATH} with {len(merged)} rows.")

with pd.option_context("display.max_colwidth", 120, "display.max_rows", 10):
    display(merged.head(10))


# Stage 7. Instert csv into tech_data_platform_work.temp_ table

In [ ]:
# === Load a CSV and write it into delta.<schema>.<table> (recreate + insert, batched) ===
import math
import os
import time
import json
import pandas as pd

# ---------------- CONFIG ----------------
INPUT_CSV = "6_tables_union_all.csv"
TARGET_CATALOG = "delta"
TARGET_SCHEMA  = "tech_data_platform_work"
TARGET_TABLE   = "astarostina_catalog"

SELECT_COLS = [
    "fqtn",
    "master_catalog",
    "master_schema",
    "base_table",
    "master_table_name",
    "master_table_type",
    "master_columns",
    "master_partitioned_by",
    "dh_dh_found",
    "dh_dh_urn",
    "dh_dh_url",
    "dh_dh_description",
    "dh_dh_editable_description",
    "dh_dh_tags",

    "dbt_unique_id",
    "dbt_description",
    "dbt_catalog",
    "dbt_schema",
    "dbt_table",
    "dbt_materialized",
    "dbt_tags",

    "dbt_unique_key",
    "dbt_partition_by",
    "dbt_depends_on_models",
    "dbt_resolved_upstream_tables",
    "dbt_depended_by_base_tables",
    "dbt_depended_by_fqtn",

    "dh_up_matches",
    "dh_up_upstream_descriptions",
    "dh_up_upstream_editable_descs",
    "dh_up_upstream_owners",
    "dh_up_upstream_tags",

    "usage_queries_cnt",
    "usage_users_cnt",
    "usage_users_list",
    "usage_catalogs",
    "usage_top_joins",
    "usage_top_columns",
    "usage_sample_queries",
    "usage_users_with_meta",
    "usage_departments",
    "usage_directions",
    "usage_groups",
    "usage_team_leads"
]

INSERT_BATCH_SIZE = 5
FORCE_VARCHAR = True
TYPE_OVERRIDES: dict[str, str] = {}

# ---------------- Helpers ----------------
def _trino_ident(s: str) -> str:
    return f'"{str(s).replace(chr(34), chr(34)*2)}"'

def _to_trino_type(series: pd.Series) -> str:
    if pd.api.types.is_bool_dtype(series): return "boolean"
    if pd.api.types.is_integer_dtype(series): return "bigint"
    if pd.api.types.is_float_dtype(series): return "double"
    s = series.dropna().astype(str)
    if not s.empty and s.str.match(r"^\d{4}-\d{2}-\d{2}$").all(): return "date"
    if not s.empty and s.str.match(r"^\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}").all(): return "timestamp"
    return "varchar"

def _make_ddl(df: pd.DataFrame) -> str:
    cols = []
    for c in SELECT_COLS:
        trino_type = "varchar" if FORCE_VARCHAR else TYPE_OVERRIDES.get(c) or _to_trino_type(df[c])
        cols.append(f"{_trino_ident(c)} {trino_type}")

    # NEW system column
    cols.append(f"{_trino_ident('etl_updated_at')} timestamp")

    return f"""
    CREATE TABLE {_trino_ident(TARGET_CATALOG)}.{_trino_ident(TARGET_SCHEMA)}.{_trino_ident(TARGET_TABLE)} (
      {', '.join(cols)}
    )
    """

def _chunk_iter(df: pd.DataFrame, chunk_size: int):
    for i in range(0, len(df), chunk_size):
        yield df.iloc[i:i+chunk_size]

def _prepare_value(v):
    return None if pd.isna(v) else str(v)

# ---------------- 1) Load CSV ----------------
assert os.path.exists(INPUT_CSV), f"File not found: {INPUT_CSV}"
df = pd.read_csv(INPUT_CSV, low_memory=False)

print(f"[input] loaded {len(df)} rows")

# ---------------- 2) Filter rows ----------------
mask = df["master_table_name"].astype(str).str.strip().ne("")
df_f = df.loc[mask, SELECT_COLS].copy()

if df_f.empty:
    raise ValueError("Nothing to load after filtering.")

# ---------------- 3) Recreate target table ----------------
cur = conn.cursor()
cur.execute(f"DROP TABLE IF EXISTS {_trino_ident(TARGET_CATALOG)}.{_trino_ident(TARGET_SCHEMA)}.{_trino_ident(TARGET_TABLE)}")
cur.execute(_make_ddl(df_f))
print("[ddl] table created")

# ---------------- 4) Insert in batches ----------------
cols_sql = ", ".join(_trino_ident(c) for c in SELECT_COLS) + ", etl_updated_at"

total = len(df_f)
inserted = 0
t0 = time.time()

for chunk in _chunk_iter(df_f, INSERT_BATCH_SIZE):
    row_placeholder = "(" + ", ".join(["?"] * len(SELECT_COLS)) + ", current_timestamp)"
    values_sql = ", ".join([row_placeholder] * len(chunk))

    sql = f"""
        INSERT INTO {_trino_ident(TARGET_CATALOG)}.{_trino_ident(TARGET_SCHEMA)}.{_trino_ident(TARGET_TABLE)}
        ({cols_sql})
        VALUES {values_sql}
    """

    params = []
    for _, row in chunk.iterrows():
        params.extend([_prepare_value(row[c]) for c in SELECT_COLS])

    cur.execute(sql, params)
    inserted += len(chunk)

    if inserted % (INSERT_BATCH_SIZE * 2) == 0 or inserted == total:
        print(f"[insert] {inserted}/{total} rows in {time.time() - t0:.1f}s")

print(f"[done] inserted {inserted} rows into {TARGET_CATALOG}.{TARGET_SCHEMA}.{TARGET_TABLE}")


# Stage 8. Generate descriptions via LLM

In [ ]:
#new 
# === LLM descriptions for data assets (short + long), with checkpoints & retries ===
import os, json, time, logging, tempfile, shutil
from pathlib import Path
from textwrap import shorten
from datetime import datetime

import pandas as pd
import requests
import re

# -------------------- CONFIG --------------------
assert "conn" in globals(), "Trino connection `conn` is not defined."

SOURCE_TABLE = "delta.tech_data_platform_work.tmp_astarostina_catalog_20250916"
FILTER_NON_EMPTY_MASTER = True

# LLM endpoint
BASE_URL   = "https://llm-service.prod.env"
MODEL      = "gpt-oss-120b"
CA_BUNDLE  = "/etc/ssl/certs/ca-certificates.crt"
LLM_TIMEOUT = 90
LLM_TEMPERATURE = 0.15

# Batching & retries
BATCH_SIZE = 10
RETRY_ATTEMPTS = 3
RETRY_BACKOFF_S = 2.0

# IO
OUT_DIR = Path("checkpoints")
OUT_DIR.mkdir(exist_ok=True)
CKPT_FILE = OUT_DIR / "llm_desc_ckpt_20250916_01.csv"
FINAL_CSV = Path("llm_descriptions.csv")

WRITE_BACK_TO_TRINO = True
TARGET_TABLE = "delta.tech_data_platform_work.tmp_astarostina_catalog_desc_20250916"

# Limits
MAX_LIST_ITEMS = 12
MAX_EACH_ITEM_LEN = 280
MAX_QUERY_LEN = 2500
MAX_PROMPT_CHARS = 20000
SYSTEM_PROMPT = "You are a data product writer. Write concisely, clearly, business-friendly, without marketing fluff. Important rules:- Do not start the description with the catalog name (e.g., Delta or Iceberg).- Begin with a human-readable title (from description fields or inferred).- Show the full technical table name in backticks after the title, e.g.: Chat alerts transcript summary – `delta.commercial_chatbot_mart.chat_alerts_transcripts_summary`- Do not mention storage format or catalog in the prose; it is only part of the technical name in backticks."

# -------------------- LOGGING --------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
log = logging.getLogger("llm-descriptions_20250916")

# -------------------- HELPERS --------------------
def _short(text: str, width: int) -> str:
    """Safe shorten with ellipsis, also flattens newlines."""
    if text is None:
        text = ""
    return shorten(str(text).replace("\n", " "), width=width, placeholder="…")

def _parse_listish(x):
    if isinstance(x, list):
        return x
    if not isinstance(x, str):
        return []
    s = x.strip()
    if not s:
        return []
    try:
        return json.loads(s)
    except Exception:
        s2 = s.strip("[]")
        parts = [p.strip().strip("'").strip('"') for p in s2.split(",") if p.strip()]
        return parts

def _cap_list(items, max_items=MAX_LIST_ITEMS, max_len=MAX_EACH_ITEM_LEN):
    out = []
    items = items or []
    for it in items[:max_items]:
        out.append(_short(it, max_len))
    return out

def _cap_query(q: str, limit=MAX_QUERY_LEN):
    if not isinstance(q, str):
        return ""
    q_norm = " ".join(q.split())
    return (q_norm[:limit] if len(q_norm) > limit else q_norm)

def _first_non_empty(*vals, default=""):
    for v in vals:
        if isinstance(v, str) and v.strip():
            return v.strip()
    return default

def _user_prompt_short(row: pd.Series) -> str:
    fqtn = row.get("fqtn") or ""
    ttype = row.get("master_table_type") or row.get("dbt_materialized") or ""
    cols  = _cap_list(_parse_listish(row.get("usage_top_columns") or row.get("columns") or []), 25, 100)
    ex_qs = _cap_list(_parse_listish(row.get("usage_sample_queries") or []), 3, MAX_QUERY_LEN)
    joins = _cap_list(_parse_listish(row.get("usage_top_joins") or []), 12, 120)
    depts = _cap_list(_parse_listish(row.get("usage_departments") or []), 6, 60)
    directions = _cap_list(_parse_listish(row.get("usage_directions") or []), 6, 60)

    dbt_desc = _first_non_empty(row.get("dbt_description") or "")
    dh_edesc = _first_non_empty(row.get("dh_dh_editable_description") or "")
    up_tables = _cap_list(_parse_listish(row.get("dbt_upstream_tables") or []), 10, 120)
    down_tables = _cap_list(_parse_listish(row.get("dbt_depended_by_base_tables") or []), 10, 120)

    up_desc   = row.get("dh_up_upstream_descriptions") or "{}"
    up_edesc  = row.get("dh_up_upstream_editable_descs") or "{}"
    # These two may arrive as JSON strings; try to parse to dicts safely:
    
    try:
        if isinstance(up_desc, str):
            up_desc = json.loads(up_desc)
    except Exception:
        up_desc = {}
    try:
        if isinstance(up_edesc, str):
            up_edesc = json.loads(up_edesc)
    except Exception:
        up_edesc = {}

    up_pairs = []
    if isinstance(up_desc, dict):
        for k, v in list(up_desc.items())[:8]:
            up_pairs.append(f"{k}: {_short(v, 240)}")
    if isinstance(up_edesc, dict):
        for k, v in list(up_edesc.items())[:6]:
            up_pairs.append(f"{k}: {_short(v, 240)}")

    tags     = _cap_list(_parse_listish(row.get("dh_dh_tags") or []), 12, 40)
    uniq_key = _cap_list(_parse_listish(row.get("dbt_unique_key") or []), 4, 60)
    part_by  = _cap_list(_parse_listish(row.get("dbt_partition_by") or []), 4, 60)

    prompt = f"""
You are a Data Product Writer. For the Data Lake object, generate a concise business-friendly description (Unity Catalog style) based on the provided data:
- Full unique table name: {fqtn}
- Table type: {ttype}
- Columns list: {cols}
- Query usage: SQL examples {ex_qs}
- Query usage: usually joined with {joins}
- Query usage: usually used by users from departments {depts}
- Query usage: usually used by users from directions {directions}
- Current short desc (if available): "{dbt_desc}"
- Existing extra desc (if available): "{dh_edesc}"
- Upstream tables: {up_tables}
- Upstream tables descriptions: {up_pairs}
- Downstream tables: {down_tables}
- Tags: {tags}
- Unique Key: {uniq_key or '<n/a>'}
- Partitioned By: {part_by or '<n/a>'}

Guidelines:
- Be concise, clear, and business-oriented.
- Avoid marketing language and vague statements.
- Mention only the sections where actual data is available. If a field is missing, skip it entirely.
- Organize the output into: short description, typical usage, usage example, common joins, upstream tables, downstream tables.
Return ONLY the paragraph.
""".strip()

    return prompt[:MAX_PROMPT_CHARS]

def _user_prompt_long(row: pd.Series) -> str:
    fqtn = row.get("fqtn") or ""
    ttype = row.get("master_table_type") or row.get("dbt_materialized") or ""
    cols  = _cap_list(_parse_listish(row.get("usage_top_columns") or row.get("columns") or []), 40, 100)
    ex_qs = _cap_list(_parse_listish(row.get("usage_sample_queries") or []), 2, MAX_QUERY_LEN)
    joins = _cap_list(_parse_listish(row.get("usage_top_joins") or []), 15, 120)
    depts = _cap_list(_parse_listish(row.get("usage_departments") or []), 8, 60)
    directions = _cap_list(_parse_listish(row.get("usage_directions") or []), 8, 60)

    dbt_desc   = _first_non_empty(row.get("dbt_description") or "")
    up_tables  = _cap_list(_parse_listish(row.get("dbt_upstream_tables") or []), 15, 120)
    down_base  = _cap_list(_parse_listish(row.get("dbt_depended_by_base_tables") or []), 15, 120)
    dh_edesc   = _first_non_empty(row.get("dh_dh_editable_description") or "")
    tags       = _cap_list(_parse_listish(row.get("dh_dh_tags") or []), 16, 40)
    uniq_key   = _cap_list(_parse_listish(row.get("dbt_unique_key") or []), 6, 60)
    part_by    = _cap_list(_parse_listish(row.get("dbt_partition_by") or []), 6, 60)

    up_desc   = row.get("dh_up_upstream_descriptions") or "{}"
    up_edesc  = row.get("dh_up_upstream_editable_descs") or "{}"

    try:
        if isinstance(up_desc, str):
            up_desc = json.loads(up_desc)
    except Exception:
        up_desc = {}
    try:
        if isinstance(up_edesc, str):
            up_edesc = json.loads(up_edesc)
    except Exception:
        up_edesc = {}

    up_pairs = []
    if isinstance(up_desc, dict):
        for k, v in list(up_desc.items())[:10]:
            up_pairs.append(f"{k}: {_short(v, 240)}")
    if isinstance(up_edesc, dict):
        for k, v in list(up_edesc.items())[:8]:
            up_pairs.append(f"{k}: {_short(v, 240)}")

    sql_block = f"\n```sql\n{ex_qs[0]}\n```\n" if ex_qs else ""

    prompt = f"""
You are a Data Product Writer. Generate a LONG business-friendly description for a Data Lake object.
Make it clear, structured, and helpful for analysts: a short overview + typical usage bullets + joins + upstream/downstream context + (optional) SQL example.

Context:
- Full unique table name: {fqtn}
- Table type: {ttype}
- Columns (sample): {cols}
- Query usage:
  - Usually joined with: {joins}
  - Directions: {directions}
- Existing short desc: "{dbt_desc}"
- Upstream tables (if provided): {up_tables}
- Upstream tables desc (combined): {up_pairs}
- Downstream tables: {down_base}
- Tags: {tags}
- Unique Key: {uniq_key or '<n/a>'}
- Partitioned By: {part_by or '<n/a>'}
- Usage examples: {sql_block}

Guidelines:
- Start with a 1–2 sentence overview. Start the overview with a human-friendly title, not with "Delta ..." or "Iceberg ...". Then show the full table name in backticks.
- Don't add "Overview" word at the beginning, just begin with the actual description.
- Then optional "# Typical usage:" bullet list (1–3 bullets).
- Then optional "# Common joins:" one-line list.
- Then optional "# Usage example:" with 1-2 SQL fenced blocks if provided.
- Then optional "# Upstream / Downstream:" sections listing relevant tables.
- Mention only the sections where actual data is available. If a field is missing, skip it entirely.
- Instead of "base table" say "table".
- Instead of "read-only view" say "view".

Return Markdown only.

Example of description: "Chatbot Interaction Analytics - commercial_chatbot_mart.cia_dialogs
Table with one record per chat conversation, including timestamps, routing events, language switches, message counts, CSAT, ownership/agent fields, and basic sanitised text for downstream analysis.
# Typical usage:
- De-duplicate and select the latest conversation per uuid (e.g., by max created_at).
- Slice performance by channel, country, issue_type/primary_issue, and language.

# Usage example:
SELECT  chat_id, messaging_session_name, chat_link,
        created_at, ended_at, country,
        issue_type, primary_issue,
        total_messages_cnt, start_language, end_language
FROM    delta.commercial_chatbot_mart.cia_dialogs
WHERE   is_empty_chat = FALSE
  AND   is_agent_interaction = TRUE
  AND   total_messages_cnt > 0
  AND   created_at BETWEEN TIMESTAMP '2025-08-05 00:00:00'
                        AND TIMESTAMP '2025-09-06 23:59:59'
  AND lower(issue_type) LIKE '%complaints%'
LIMIT  100;

# Common joins: ['commercial_chatbot_core_sens.chat_cmp', 'commercial_mart.country_dim', 'commercial_chatbot_mart.v_vertica_dm_interaction_issue_type_dim', 'commercial_chatbot_mart.v_vertica_dm_interaction_issue_category_dim', 'commercial_chatbot_core_sens.chat_alerts_transcripts_summary', 'trading_core.clickhouse_xdata_user_dim']
# Unique Key: e.g. ['chat_id'] if the unique key is available
# Partitioned By: <if available - partitioned by keys>
# Used By: <if available - directions comma separated>
# Upstream tables: ['stg.sf_messaging_session', 'commercial_chatbot_core.chat_csat', 'commercial_chatbot_core.chat_languages', 'commercial_chatbot_core_sens.chat_messages', 'commercial_chatbot_core.chat_salesforce_transferring', 'relationship_mart.vertica_sse_client_lfact', 'commercial_chatbot_core_sens.chat_metadata']
# Downstream tables: ['commercial_chatbot_mart.cia_dialogs_vertica_out', 'commercial_chatbot_mart.rm_support_requests_last]
""".strip()

    return prompt[:MAX_PROMPT_CHARS]

def _llm_call(system_msg: str, user_msg: str) -> str:
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": user_msg},
        ],
        "temperature": LLM_TEMPERATURE,
        "stream": False,
    }
    for attempt in range(1, RETRY_ATTEMPTS+1):
        try:
            r = requests.post(f"{BASE_URL}/v1/chat/completions",
                              headers={"Content-Type": "application/json"},
                              json=payload,
                              timeout=LLM_TIMEOUT,
                              verify=CA_BUNDLE)
            if r.status_code == 200:
                js = r.json()
                return js["choices"][0]["message"]["content"]
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(RETRY_BACKOFF_S * attempt)
                continue
            raise RuntimeError(f"HTTP {r.status_code}: {r.text[:400]}")
        except Exception:
            if attempt == RETRY_ATTEMPTS:
                raise
            time.sleep(RETRY_BACKOFF_S * attempt)
    return ""

def _atomic_append_row(row_dict: dict, path: Path, header_cols: list[str]):
    exists = path.exists() and path.stat().st_size > 0
    df = pd.DataFrame([row_dict], columns=header_cols)
    if exists:
        with open(path, "a", encoding="utf-8") as f:
            df.to_csv(f, index=False, header=False)
    else:
        df.to_csv(path, index=False)

# --- helper to derive short desc from long desc ---
def _extract_short_from_markdown(md: str) -> str:
    """
    Take the first meaningful paragraph from an LLM markdown response:
    - strip fenced code blocks
    - split on blank lines
    - skip markdown headers
    - collapse whitespace to a single line
    """
    if not isinstance(md, str) or not md.strip():
        return ""
    # remove fenced code blocks
    text = re.sub(r"```.*?```", "", md, flags=re.S)
    # split into paragraphs by blank lines
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    # choose first non-header paragraph
    for p in paras:
        if not p.lstrip().startswith("#"):
            return " ".join(p.split())
    # fallback: first paragraph (even if header)
    return " ".join(paras[0].split()) if paras else ""

# --- NEW helper: micro-shorten a short description via LLM, with fallback ---
def _shorten_with_llm(short_desc: str, max_chars: int = 240) -> str:
    """
    Ask LLM to rewrite short_desc to a single, business-friendly sentence
    within max_chars. If the call fails, we fall back to a hard trim.
    """
    if not isinstance(short_desc, str) or not short_desc.strip():
        return ""

    user_msg = (
        "Rewrite the paragraph below as ONE concise, clear, business-friendly sentence in English. "
        f"Keep key meaning. Limit to ≤ {max_chars} characters. No marketing, no emojis.\n\n"
        f"Text:\n{short_desc}"
    )

    try:
        out = _llm_call(SYSTEM_PROMPT, user_msg).strip()
        # hard safety cap if the model returns longer text
        if len(out) > max_chars:
            out = out[:max_chars].rstrip()
        return out
    except Exception:
        # fallback: hard trim the original
        return short_desc[:max_chars].rstrip()


# -------------------- LOAD DATA --------------------
q = f"""
SELECT
   fqtn, master_catalog, master_schema, base_table, master_table_name, master_table_type,
   dh_dh_found, dh_dh_urn, dh_dh_url, dh_dh_description, dh_dh_editable_description, dh_dh_tags,
   dbt_unique_id, dbt_description, dbt_catalog, dbt_schema, dbt_table, dbt_materialized, dbt_tags,
   dbt_unique_key, dbt_partition_by, dbt_depends_on_models, dbt_depended_by_base_tables, dbt_depended_by_fqtn
   , dh_up_upstream_descriptions
   , dh_up_upstream_editable_descs
   , dh_up_upstream_owners
   , dh_up_upstream_tags
   ,
   usage_queries_cnt, usage_users_cnt, usage_users_list, usage_catalogs,
   usage_top_joins, usage_top_columns, usage_sample_queries,
   usage_users_with_meta, usage_departments, usage_directions, usage_groups, usage_team_leads
FROM {SOURCE_TABLE}
WHERE lower(master_schema) in ( 'commercial_chatbot_mart','commercial_chatbot_core', 'trading_core', 'partnership_core','partnership_mart', 'trading_mart') or base_table like '%vertica%' or base_table like '%clickhouse%' or base_table like '%pg_%' 
OR lower(base_table) in ( 'marketing_core.v_amplitude_event', 'marketing_core.bq_ga_base_events')
"""
cur = conn.cursor(); cur.execute(q)
rows = cur.fetchall()
cols = [c[0] for c in cur.description]
df = pd.DataFrame(rows, columns=cols)
if FILTER_NON_EMPTY_MASTER:
    df = df[df["master_table_name"].astype(str).str.strip().ne("")].copy()

log.info(f"[input] loaded {len(df)} rows from {SOURCE_TABLE}")

# -------------------- RESUME FROM CHECKPOINT --------------------
ckpt_cols = ["fqtn","short_desc","shorten_by_llm","long_desc","status","error","generated_at"]

if CKPT_FILE.exists() and CKPT_FILE.stat().st_size > 0:
    ckpt = pd.read_csv(CKPT_FILE)
    done = set(ckpt["fqtn"].astype(str))
else:
    done = set()
    pd.DataFrame(columns=ckpt_cols).to_csv(CKPT_FILE, index=False)

todo = df[~df["fqtn"].astype(str).isin(done)].reset_index(drop=True)
log.info(f"[todo] to generate: {len(todo)} (already in ckpt: {len(done)})")

# -------------------- GENERATE --------------------
sys_msg = SYSTEM_PROMPT
count = 0
for _, row in todo.iterrows():
    fqtn = str(row.get("fqtn") or "")
    try:
        # 1) generate long description
        user_long  = _user_prompt_long(row)
        long_desc  = _llm_call(sys_msg, user_long)

        # 2) derive short from first paragraph of the long desc
        short_desc = _extract_short_from_markdown(long_desc)

        # 3) micro-shorten with LLM (extra field)
        shorten_by_llm = _shorten_with_llm(short_desc, max_chars=240)

        _atomic_append_row({
            "fqtn": fqtn,
            "short_desc": short_desc,
            "shorten_by_llm": shorten_by_llm,
            "long_desc": long_desc,
            "status": "ok",
            "error": "",
            "generated_at": datetime.utcnow().isoformat(timespec="seconds") + "Z"
        }, CKPT_FILE, ckpt_cols)

    except Exception as e:
        _atomic_append_row({
            "fqtn": fqtn,
            "short_desc": "",
            "shorten_by_llm": "",
            "long_desc": "",
            "status": "error",
            "error": str(e)[:500],
            "generated_at": datetime.utcnow().isoformat(timespec="seconds") + "Z"
        }, CKPT_FILE, ckpt_cols)

    count += 1
    if count % 10 == 0:
        log.info(f"[progress] {count}/{len(todo)}")

# -------------------- SAVE FINAL CSV --------------------
final = pd.read_csv(CKPT_FILE, low_memory=False)
final.to_csv(FINAL_CSV, index=False)
log.info(f"[done] wrote {FINAL_CSV} with {len(final)} rows")



# --- Sanitize & write-back ---
def _to_str(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    return str(x)

ins = pd.read_csv("llm_descriptions.csv")
ins["fqtn"]            = ins["fqtn"].apply(_to_str)
ins["short_desc"]      = ins["short_desc"].apply(_to_str)
ins["shorten_by_llm"]  = ins.get("shorten_by_llm", "").apply(_to_str)
ins["long_desc"]       = ins["long_desc"].apply(_to_str)

if "generated_at" not in ins.columns:
    ins["generated_at"] = pd.Timestamp.utcnow()
else:
    ins["generated_at"] = pd.to_datetime(ins["generated_at"], errors="coerce", utc=True)

cur = conn.cursor()
cur.execute(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
cur.execute(f"""
    CREATE TABLE {TARGET_TABLE} (
        fqtn VARCHAR,
        short_desc VARCHAR,
        shorten_by_llm VARCHAR,
        long_desc VARCHAR,
        generated_at TIMESTAMP
    )
""")

INSERT_BATCH_SIZE = 10
total = len(ins)
inserted = 0

for start in range(0, total, INSERT_BATCH_SIZE):
    chunk = ins.iloc[start:start+INSERT_BATCH_SIZE]
    placeholders = ",".join(
        ["(CAST(? AS VARCHAR), CAST(? AS VARCHAR), CAST(? AS VARCHAR), CAST(? AS VARCHAR), CAST(? AS TIMESTAMP WITH TIME ZONE))"]
        * len(chunk)
    )
    sql = f"""
        INSERT INTO {TARGET_TABLE} (fqtn, short_desc, shorten_by_llm, long_desc, generated_at)
        VALUES {placeholders}
    """
    params = []
    for _, r in chunk.iterrows():
        params.extend([
            r["fqtn"],
            r["short_desc"],
            r["shorten_by_llm"],
            r["long_desc"],
            r["generated_at"].to_pydatetime() if pd.notna(r["generated_at"]) else None,
        ])
    cur.execute(sql, params)
    inserted += len(chunk)
    print(f"[write-back] inserted {inserted}/{total}")

print(f"[done] inserted {inserted} rows into {TARGET_TABLE}")
